# cenario 3 - avg com valores contínuos recebe input continuo->spsocieos finais continuas (treino ainda é o dataset da grid do class 3)

In [1]:
import random
import numpy as np
import math
from deap import base, creator, tools, algorithms
import matplotlib.pyplot as plt  # Importar matplotlib para plotar gráficos
from functools import partial  # Adicionar no início do código
import networkx as nx
from link_capacity import calcular_capacidade_link, interference_from_jammer
from antenna_null import realistic_antenna_gain

NUM_UAVS = 4
lambda_0 = 0.125
P_INTERFERENCE_DBM = 100
P_NOISE_DBM = -100
BANDWIDTH = 20e6
TRANSMIT_POWER_DBM = 20
MINDIST = 20

# Definir os tipos de indivíduos e fitness
creator.create("FitnessMax", base.Fitness, weights=(1.0,))  # Maximizar o fitnessAS
creator.create("Individual", list, fitness=creator.FitnessMax)  # Indivíduo é uma listaas

#ATENCAO JAMMER POSITION NO CREATE INDIVIDUAL

# Nova função para criar um indivíduo
def create_individual(num_uavs, num_timeslots, timeslot_length, epoch, min_y, max_y, epoch_total_length=200, previous_best_solution=None, initial_positions=None, jammer_position=None):
    individual = []
    
    # Definir as posições iniciais de acordo com o epoch
    if epoch == 0:
        start_positions = initial_positions  # Posições fixas passadas manualmente
    else:
        start_positions = []
        for uav in range(num_uavs):
            last_timeslot_idx = (num_timeslots - 1) * num_uavs * 2 + uav * 2
            start_positions.append(previous_best_solution[last_timeslot_idx:last_timeslot_idx + 2])
    
    # Gerar posições finais adaptado à tua lógica
    final_positions = []
    for uav in range(num_uavs):
        if epoch == 0:
            # Epoch 1: 0 → 200 (epoch_length)
            last_x = random.uniform(epoch_total_length - timeslot_length, epoch_total_length)
        else:
            # Epoch 2: 100 → 300, Epoch 3: 200 → 400, etc.
            end_position = epoch_total_length + (epoch * timeslot_length)
            last_x = random.uniform(end_position - timeslot_length, end_position)
        
        last_y = random.uniform(min_y, max_y)
        final_positions.append([last_x, last_y])
    
    # Calcular todas as posições intermediárias
    for t in range(num_timeslots):
        for uav in range(num_uavs):
            start_x, start_y = start_positions[uav]
            end_x, end_y = final_positions[uav]
            
            # Calcular posição baseada no progresso linear
            alpha = t / (num_timeslots - 1) if num_timeslots > 1 else 0
            
            x = start_x + alpha * (end_x - start_x)
            y = start_y + alpha * (end_y - start_y)
            
            # Adicionar x,y ao indivíduo
            individual.extend([x, y])
    
    return creator.Individual(individual)


def print_communication_values_per_timeslot(individual, num_uavs, num_timeslots, jammer_position):
    all_min_capacities = []
    all_interference_matrices = []
    all_fitness_values = []  # ← ADICIONAR para armazenar fitness de cada timeslot

    for t in range(num_timeslots):
        angles = []
        positions = []
        
        for i in range(num_uavs):
            idx = t * num_uavs * 2 + i * 2
            x = individual[idx]
            y = individual[idx + 1]
            
            angle = np.degrees(np.arctan2(jammer_position[1] - y, jammer_position[0] - x))
            positions.append(np.array([x, y]))
            angles.append(angle)

        comm_matrix, interference_matrix = calculate_communication_capacity(angles, [1.0] * num_uavs, positions, jammer_position)
        all_interference_matrices.append(interference_matrix)

        # ← USAR A MESMA LÓGICA DO evaluate_individual
        # Avaliar grafo
        G = nx.DiGraph()
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j and comm_matrix[i][j] > 0:
                    G.add_edge(i, j, capacity=comm_matrix[i][j])

        # Encontrar links usados através dos caminhos mínimos
        links_usados = set()
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j:
                    try:
                        path = nx.shortest_path(G, source=i, target=j, weight=lambda u, v, d: 1/d['capacity'])
                        for u, v in zip(path[:-1], path[1:]):
                            links_usados.add((u, v))
                            links_usados.add((v, u))
                    except:
                        pass
        
        # Cálculo usando apenas links usados (IGUAL AO evaluate_individual)
        capacidades_usadas = [comm_matrix[u][v] for u, v in links_usados if u < num_uavs and v < num_uavs]
        if capacidades_usadas:
            C_media_total = np.mean(capacidades_usadas)
            C_min_total = min(capacidades_usadas)
            all_min_capacities.extend(capacidades_usadas)  # ← Adicionar todas as capacidades usadas
        else:
            C_media_total = 0
            C_min_total = 0
        
        # Calcular fitness do timeslot (IGUAL AO evaluate_individual)
        alpha, beta = 1.0, 1.0
        fitness = (C_media_total ** alpha) * (C_min_total ** beta)
        all_fitness_values.append(fitness)

    # Calcular fitness médio (IGUAL AO evaluate_individual)
    if len(all_fitness_values) > 0:
        avg_fitness = sum(all_fitness_values) / len(all_fitness_values)
    else:
        avg_fitness = 0.0

    if len(all_min_capacities) > 0:
        avg_min_capacity = sum(all_min_capacities) / len(all_min_capacities)
    else:
        avg_min_capacity = 0.0

    return all_min_capacities, all_interference_matrices, avg_min_capacity, avg_fitness  # ← ADICIONAR avg_fitness

# Função para calcular a capacidade de comunicação entre todos os UAVs
def calculate_communication_capacity(antenna_angles, alignments, positions, jammer_position):
    communication_capacity = np.zeros((NUM_UAVS, NUM_UAVS))
    interference_matrix = np.full((NUM_UAVS, NUM_UAVS), P_INTERFERENCE_DBM)

    for i in range(NUM_UAVS):
        null_dir_i = antenna_angles[i]

        dx_jam = jammer_position[0] - positions[i][0]
        dy_jam = jammer_position[1] - positions[i][1]
        dir_jammer_to_uav = np.degrees(np.arctan2(dy_jam, dx_jam)) % 360

        G_jammer_i = realistic_antenna_gain(dir_jammer_to_uav, null_dir_i)

        dist_jammer_i = np.linalg.norm(positions[i] - jammer_position)
        P_interf_dBm_i = interference_from_jammer(P_INTERFERENCE_DBM, G_jammer_i, lambda_0, dist_jammer_i)

        interference_matrix[i, :] = P_interf_dBm_i

    for i in range(NUM_UAVS):
        for j in range(NUM_UAVS):
            if i != j:
                null_dir_i = antenna_angles[i]
                null_dir_j = antenna_angles[j]

                dx_ij = positions[j][0] - positions[i][0]
                dy_ij = positions[j][1] - positions[i][1]
                dir_tx_to_rx = np.degrees(np.arctan2(dy_ij, dx_ij)) % 360

                dx_ji = positions[i][0] - positions[j][0]
                dy_ji = positions[i][1] - positions[j][1]
                dir_rx_to_tx = np.degrees(np.arctan2(dy_ji, dx_ji)) % 360

                G_tx = realistic_antenna_gain(dir_tx_to_rx, null_dir_i)
                G_rx = realistic_antenna_gain(dir_rx_to_tx, null_dir_j)

                capacity = calcular_capacidade_link(
                    pos1=positions[i],
                    pos2=positions[j],
                    lambda_0=lambda_0,
                    P_tx_dBm=TRANSMIT_POWER_DBM,
                    G_tx_dB=G_tx,
                    G_rx_dB=G_rx,
                    P_interference_dBm=interference_matrix[i, j],
                    P_noise_dBm=P_NOISE_DBM,
                    bandwidth=BANDWIDTH
                )

                communication_capacity[i][j] = capacity

    return communication_capacity, interference_matrix

def has_collision(individual, num_uavs, num_timeslots, min_distance=MINDIST):
    """
    Verifica se há pelo menos uma colisão nas posições dos timeslots.
    Para na primeira colisão encontrada.
    
    Args:
        individual: Lista com as posições [x1, y1, x2, y2, ...]
        num_uavs: Número de UAVs
        num_timeslots: Número de timeslots
        min_distance: Distância mínima de segurança
    
    Returns:
        bool: True se há pelo menos uma colisão, False caso contrário
    """
    # Verificar cada timeslot
    for t in range(num_timeslots):
        # Extrair posições de todos os UAVs neste timeslot
        positions = []
        for uav in range(num_uavs):
            idx_x = t * num_uavs * 2 + uav * 2
            idx_y = t * num_uavs * 2 + uav * 2 + 1
            x = individual[idx_x]
            y = individual[idx_y]
            positions.append(np.array([x, y]))
        
        # Verificar colisões entre todos os pares de UAVs neste timeslot
        for i in range(num_uavs):
            for j in range(i + 1, num_uavs):
                distance = np.linalg.norm(positions[i] - positions[j])
                if distance < min_distance:
                    return True  # PARA IMEDIATAMENTE na primeira colisão
    
    return False  # Nenhuma colisão encontrada

def evaluate_individual(individual, num_uavs, num_timeslots, jammer_position):
    # Verificar se há pelo menos uma colisão
    if has_collision(individual, num_uavs, num_timeslots):
        return (0.0,)  # FITNESS 0 imediatamente
    
    # Se não há colisões, calcular o fitness normalmente
    alpha, beta = 1.0, 1.0
    total_fitness = 0

    for t in range(num_timeslots):
        positions = []
        angles = []
        for i in range(num_uavs):
            idx = t * num_uavs * 2 + i * 2
            x = individual[idx]
            y = individual[idx + 1]
            
            # Calcular ângulo dinamicamente
            angle = np.degrees(np.arctan2(jammer_position[1] - y, jammer_position[0] - x))
            positions.append(np.array([x, y]))
            angles.append(angle)

        # Calcular matriz de comunicação
        comm_matrix, _ = calculate_communication_capacity(angles, [1.0] * num_uavs, positions, jammer_position)

        # Avaliar grafo
        G = nx.DiGraph()
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j and comm_matrix[i][j] > 0:
                    G.add_edge(i, j, capacity=comm_matrix[i][j])

        # Encontrar links usados através dos caminhos mínimos
        links_usados = set()
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j:
                    try:
                        path = nx.shortest_path(G, source=i, target=j, weight=lambda u, v, d: 1/d['capacity'])
                        # Adicionar links usados
                        for u, v in zip(path[:-1], path[1:]):
                            links_usados.add((u, v))
                            links_usados.add((v, u))  # Adicionar a aresta reversa
                    except:
                        pass
        
        # Cálculo do fitness usando apenas links usados
        capacidades_usadas = [comm_matrix[u][v] for u, v in links_usados if u < num_uavs and v < num_uavs]
        if capacidades_usadas:
            C_media_total = np.mean(capacidades_usadas)
            C_min_total = min(capacidades_usadas)
        else:
            C_media_total = 0
            C_min_total = 0
        
        # Fitness sem penalização por colisões (já sabemos que não há colisões)
        fitness = (C_media_total ** alpha) * (C_min_total ** beta)
        total_fitness += fitness

    # Calcular a média do fitness total
    average_fitness = total_fitness / num_timeslots if num_timeslots > 0 else 0
    return (average_fitness,)

def mutate_individual(individual, min_x, max_x, min_y, max_y, mutation_rate, num_uavs, num_timeslots, epoch, timeslot_length, epoch_total_length):
    for uav in range(num_uavs):
        if random.random() < mutation_rate:
            # Extrair posições iniciais
            start_x = individual[uav * 2]
            start_y = individual[uav * 2 + 1]
            
            # Nova posição final - ADAPTADO À TUA LÓGICA
            if epoch == 0:
                # Epoch 1: 0 → 200 (epoch_length)
                new_final_x = random.uniform(epoch_total_length - timeslot_length, epoch_total_length)
            else:
                # Epoch 2: 100 → 300, Epoch 3: 200 → 400, etc.
                end_position = epoch_total_length + (epoch * timeslot_length)
                new_final_x = random.uniform(end_position - timeslot_length, end_position)
            
            new_final_y = random.uniform(min_y, max_y)
            
            # Atualizar a posição final
            individual[(num_timeslots - 1) * num_uavs * 2 + uav * 2] = new_final_x
            individual[(num_timeslots - 1) * num_uavs * 2 + uav * 2 + 1] = new_final_y
            
            # Recalcular posições intermediárias (t=1 até num_timeslots-1)
            for t in range(1, num_timeslots):
                alpha = t / (num_timeslots - 1) if num_timeslots > 1 else 0
                x = start_x + alpha * (new_final_x - start_x)
                y = start_y + alpha * (new_final_y - start_y)
                
                idx_x = t * num_uavs * 2 + uav * 2
                idx_y = t * num_uavs * 2 + uav * 2 + 1
                individual[idx_x] = x
                individual[idx_y] = y
    
    return individual,


def custom_crossover(ind1, ind2, num_uavs):
    num_timeslots = len(ind1) // (num_uavs * 2)
    
    # Trocar posições de UAVs inteiros (exceto a posição inicial t=0)
    for uav in range(num_uavs):
        if random.random() < 0.5:
            for t in range(1, num_timeslots):  # Começar de t=1
                idx_x = t * num_uavs * 2 + uav * 2
                idx_y = t * num_uavs * 2 + uav * 2 + 1
                
                # Trocar x,y para todos os timeslots deste UAV (exceto t=0)
                ind1[idx_x], ind2[idx_x] = ind2[idx_x], ind1[idx_x]
                ind1[idx_y], ind2[idx_y] = ind2[idx_y], ind1[idx_y]
    
    return ind1, ind2

# Configuração do DEAP atualizada
def setup_deap(num_uavs, num_timeslots, timeslot_length, epoch, min_y, max_y, crossover_rate, mutation_rate, initial_positions, previous_best_solution=None, jammer_position=None, epoch_total_length=300):
    toolbox = base.Toolbox()
    
    # Registrar funções
    toolbox.register("individual", create_individual, 
                     num_uavs=num_uavs, 
                     num_timeslots=num_timeslots, 
                     timeslot_length=timeslot_length, 
                     epoch=epoch, 
                     min_y=min_y, 
                     max_y=max_y,
                     epoch_total_length=epoch_total_length,  # ← ADICIONAR
                     previous_best_solution=previous_best_solution,
                     initial_positions=initial_positions,
                     jammer_position=jammer_position)  # Passar a posição do jammer aqui
    
    # O restante do código permanece o mesmo...

    
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    toolbox.register("evaluate", evaluate_individual, num_uavs=num_uavs, num_timeslots=num_timeslots, jammer_position=jammer_position)
    
    # Usar partial para fixar o argumento num_uavs na função custom_crossover
    toolbox.register("mate", partial(custom_crossover, num_uavs=num_uavs))
    
    min_x = epoch * num_timeslots * timeslot_length
    max_x = (epoch * num_timeslots + num_timeslots) * timeslot_length
    
    toolbox.register("mutate", mutate_individual, 
                 min_x=min_x, 
                 max_x=max_x, 
                 min_y=min_y, 
                 max_y=max_y, 
                 mutation_rate=mutation_rate,
                 num_uavs=num_uavs,
                 num_timeslots=num_timeslots,
                 epoch=epoch,
                 timeslot_length=timeslot_length,
                 epoch_total_length=epoch_total_length)  # ← ADICIONAR
    
    toolbox.register("select", tools.selTournament, tournsize=3)  # Seleção por torneio
    
    return toolbox

def genetic_algorithm(num_uavs, num_timeslots, timeslot_length, population_size, generations, crossover_rate, mutation_rate, epoch, initial_positions, previous_best_solution=None, jammer_position=None, min_y=0.0, max_y=5.0, epoch_total_length=300):
    # Configurar o DEAP
    toolbox = setup_deap(
        num_uavs=num_uavs,
        num_timeslots=num_timeslots,
        timeslot_length=timeslot_length,
        epoch=epoch,
        min_y=min_y,
        max_y=max_y,
        crossover_rate=crossover_rate,
        mutation_rate=mutation_rate,
        initial_positions=initial_positions,
        previous_best_solution=previous_best_solution,
        jammer_position=jammer_position,
        epoch_total_length=epoch_total_length  # ← ADICIONAR
    )

    
    # Criar população inicial
    population = toolbox.population(n=population_size)
    
    # Avaliar a população inicial
    fitnesses = list(map(toolbox.evaluate, population))
    for ind, fit in zip(population, fitnesses):
        ind.fitness.values = fit
    
    # Configurar estatísticas para impressão
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("std", np.std)
    stats.register("min", np.min)
    stats.register("max", np.max)
    
    # Listas para armazenar os dados de cada geração
    gen_list = []
    avg_list = []
    std_list = []
    min_list = []
    max_list = []
    
    # Executar o algoritmo genético com eaSimple
    for gen in range(generations):
        # Avançar uma geração
        algorithms.eaSimple(
            population, 
            toolbox, 
            cxpb=crossover_rate,  # Probabilidade de cruzamento
            mutpb=mutation_rate,  # Probabilidade de mutação
            ngen=1,               # Apenas uma geração por iteração
            stats=stats,          # Estatísticas para impressão
            verbose=False         # Desativar impressão da tabela para cada geração
        )
        
        # Coletar os dados da geração atual
        record = stats.compile(population)
        gen_list.append(gen)
        avg_list.append(record["avg"])
        std_list.append(record["std"])
        min_list.append(record["min"])
        max_list.append(record["max"])
        
        # Escrever os valores no arquivo
        #write_fitness_values(epoch, gen, record["avg"], record["max"], record["min"], record["std"])
    
    # Retornar o melhor indivíduo
    best_individual = tools.selBest(population, k=1)[0]
    return best_individual

# Função para gerar posições iniciais dos UAVs
def generate_initial_positions(num_uavs, min_y, max_y, timeslot_length, manual=False, manual_positions=None):
    if manual and manual_positions is not None:
        return manual_positions  # Usa as posições fornecidas

    positions = []
    max_attempts = 1000  # para evitar loops infinitos

    for _ in range(num_uavs):
        attempts = 0
        while True:
            y = random.uniform(min_y, max_y)
            x = random.uniform(0, timeslot_length)  # entre -timeslot_length e 0
            
            candidate = (x, y)
            
            # Verifica se está longe o suficiente das outras posições já geradas
            if all(np.linalg.norm(np.array(candidate) - np.array(pos)) >= MINDIST for pos in positions):
                positions.append(candidate)
                break
            
            attempts += 1
            if attempts >= max_attempts:
                # Se não conseguir, aceita a posição mesmo assim para evitar bloqueio
                positions.append(candidate)
                break
                
    return positions

# def save_best_solution_to_file(best_solution, initial_positions, filename="best_solution_epoch.txt"):
#     with open(filename, "a") as file:
#         # Escrever apenas a melhor solução (que já inclui tudo)
#         individual_str = ' '.join([f"{best_solution[i]:.2f}" for i in range(len(best_solution))])
#         file.write(f"{individual_str}\n \n")

def save_best_solution_to_file(best_solution, initial_positions, filename="best_solution_epoch.txt"):
    # Ler o conteúdo existente do arquivo
    try:
        with open(filename, "r") as file:
            existing_content = file.readlines()
            # Processar o conteúdo existente se necessário
            print("Conteúdo existente no arquivo:")
            for line in existing_content:
                print(line.strip())
    except FileNotFoundError:
        # Se o arquivo não existir, não há conteúdo para ler
        print("O arquivo não existe. Criando um novo arquivo.")

    # Escrever a nova melhor solução no final do arquivo
    with open(filename, "a") as file:
        individual_str = ' '.join([f"{best_solution[i]:.2f}" for i in range(len(best_solution))])
        file.write(f"{individual_str}\n\n")


def write_fitness_values(epoch, gen, avg, max_val, min_val, std, filename="fitness_values.txt"):
    
    with open(filename, "a") as file:
        if gen == 0:  # Escrever o cabeçalho no início de cada epoch
            file.write(f"=== Epoch {epoch + 1} ===\n")
            file.write("gen\tavg\tmax\tmin\tstd\n")
        file.write(f"{gen}\t{avg:.4f}\t{max_val:.4f}\t{min_val:.4f}\t{std:.4f}\n")

# Função principal atualizada
def simulate_uavs_with_ga(num_epochs, num_timeslots, timeslot_length, num_uavs, population_size=50, generations=50, crossover_rate=0.8, mutation_rate=0.15, manual_initial_positions=None, jammer_position=None, min_y=0.0, max_y=10.0, epoch_total_length=300):
    # Gerar posições iniciais dos UAVs
    initial_positions = generate_initial_positions(
        num_uavs, min_y, max_y, timeslot_length,
        manual=manual_initial_positions is not None,
        manual_positions=manual_initial_positions
    )

    previous_best_solution = None
    
    for epoch in range(num_epochs):
        # Executar o algoritmo genético
        best_solution = genetic_algorithm(
            num_uavs=num_uavs,
            num_timeslots=num_timeslots,
            timeslot_length=timeslot_length,
            population_size=population_size,
            generations=generations,
            crossover_rate=crossover_rate,
            mutation_rate=mutation_rate,
            epoch=epoch,
            initial_positions=initial_positions,
            previous_best_solution=previous_best_solution,
            jammer_position=jammer_position,  # Passar a posição do jammer aqui
            min_y=min_y,
            max_y=max_y,
            epoch_total_length=epoch_total_length  # ← ADICIONAR
        )

        # O restante do código permanece o mesmo...


        # Salvar a melhor solução em um arquivo de texto
        #save_best_solution_to_file(best_solution, initial_positions)
        
        all_min_capacities, all_interference_matrices, avg_min_capacity, avg_fitness = print_communication_values_per_timeslot(best_solution, num_uavs, num_timeslots, jammer_position)

        # Guardar a melhor solução para a próxima epoch


        # # Calcular e imprimir a coerência dos valores
        # best_fitness = best_solution.fitness.values[0]
        # recalculated_fitness = evaluate_individual(best_solution, num_uavs, num_timeslots, jammer_position)[0]
        # # Ler o último valor máximo do fitness_values.txt
        # with open("fitness_values.txt", "r") as file:
        #     lines = file.readlines()
        #     last_max_fitness = float(lines[-1].split("\t")[2])  # Último max na última linha
        # # Comparar os valores
        # print(f"Fitness da melhor solução: {best_fitness:.4f}")
        # print(f"Fitness recalculado: {recalculated_fitness:.4f}")
        # print(f"Último max no arquivo: {last_max_fitness:.4f}")
        # print(f"São iguais? {'✅' if abs(best_fitness - last_max_fitness) < 1 else '❌'}")
        # # Guardar a melhor solução para a próxima epoch
        # previous_best_solution = best_solution



        previous_best_solution = best_solution

    return best_solution, initial_positions, jammer_position, all_min_capacities, all_interference_matrices, avg_min_capacity


In [5]:
# Posições que respeitam MINDIST = 5 metros
posicoes_manuais1 = [
    (10.172259033584051,85.75523376648344),   # UAV 1
    (75.88992564121607,16.084203954794695),   # UAV 2 (60m de distância do UAV1)
    (16.084203954794695,31.066957304533616),   # UAV 3 (60m de distância do UAV1)
    (69.18455395110837,76.3309773445626)    # UAV 4 (60m de distância dos outros)
]

# Executar simulação
best_solution, initial_positions, jammer_position, all_min_capacities, all_interference_matrices, avg_min_capacity = simulate_uavs_with_ga(
    num_epochs=1, 
    num_timeslots=6, 
    timeslot_length=60, 
    num_uavs=NUM_UAVS,
    population_size=50,
    generations=50,
    manual_initial_positions=posicoes_manuais1,
    jammer_position=np.array([0,500]),
    min_y=0.0,
    max_y=60.0,
    epoch_total_length=120
)

print("Posições iniciais usadas:", initial_positions)
print("Melhor solução encontrada:", best_solution)
print("Posição do jammer:", jammer_position)

O arquivo não existe. Criando um novo arquivo.
Posições iniciais usadas: [(10.172259033584051, 85.75523376648344), (75.88992564121607, 16.084203954794695), (16.084203954794695, 31.066957304533616), (69.18455395110837, 76.3309773445626)]
Melhor solução encontrada: [10.172259033584051, 85.75523376648344, 75.88992564121607, 16.084203954794695, 16.084203954794695, 31.066957304533616, 69.18455395110837, 76.3309773445626, 28.244304626302146, 76.98461934837488, 83.51396293083637, 15.43321815219194, 26.09308300133813, 28.37897457854591, 73.66574093122235, 64.88317689349381, 46.31635021902024, 68.21400493026631, 91.13800022045669, 14.782232349589183, 36.10196204788156, 25.690991852558206, 78.14692791133632, 53.435376442425024, 64.38839581173832, 59.44339051215776, 98.762037510077, 14.131246546986427, 46.110841094424984, 23.0030091265705, 82.6281148914503, 41.98757599135624, 82.46044140445643, 50.67277609404919, 106.38607479969731, 13.48026074438367, 56.119720140968425, 20.315026400582795, 87.10

# analisar

In [6]:
def analisar_comunicacoes_detalhado(filename, num_uavs, num_timeslots, jammer_position):
    def parse_solution_file(filename):
        with open(filename, 'r') as f:
            lines = [line.strip() for line in f if line.strip()]
        
        solutions = []
        current_solution = []
        for line in lines:
            if line.startswith('==='):  # Nova época
                if current_solution:
                    solutions.append(current_solution)
                    current_solution = []
            else:
                current_solution.extend(map(float, line.split()))
        if current_solution:
            solutions.append(current_solution)
        return solutions

    def avaliar_grafo(comm_matrix):
        G = nx.DiGraph()
        
        # 1. Adicionar arestas bidirecionais com capacidades
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j and comm_matrix[i][j] > 0:
                    G.add_edge(i, j, capacity=comm_matrix[i][j])
        
        # 2. Cálculos de métricas
        metricas = {
            'capacidades': [],
            'caminhos_minimos': {},
            'bottlenecks': {},
            'conectividade': None,
            'links_usados': set(),
            'links_nao_usados': set()
        }
        
        # Preencher métricas
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j:
                    # Calcular caminhos mínimos (1/capacity como peso)
                    try:
                        path = nx.shortest_path(G, source=i, target=j, weight=lambda u, v, d: 1/d['capacity'])
                        capacidade_min = min(G[u][v]['capacity'] for u, v in zip(path[:-1], path[1:]))
                        
                        metricas['caminhos_minimos'][(i,j)] = {
                            'path': path,
                            'capacidade': capacidade_min
                        }
                        
                        # Adicionar links usados
                        for u, v in zip(path[:-1], path[1:]):
                            metricas['links_usados'].add((u, v))
                            metricas['links_usados'].add((v, u))  # Adicionar a aresta reversa
                    except:
                        pass
        
        # Calcular bottlenecks para cada nó
        for node in G.nodes():
            metricas['bottlenecks'][node] = min(
                [d['capacity'] for _, _, d in G.edges(node, data=True)],
                default=0
            )
        
        # Verificar conectividade
        metricas['conectividade'] = nx.is_strongly_connected(G)
        
        # Coletar todas as capacidades
        metricas['capacidades'] = [d['capacity'] for _, _, d in G.edges(data=True)]
        
        # Identificar links não usados
        all_links = {(i, j) for i in range(num_uavs) for j in range(num_uavs) if i != j}
        metricas['links_nao_usados'] = all_links - metricas['links_usados']
        
        return metricas

    # Processar arquivo de saída
    solucoes = parse_solution_file(filename)
    
    resultados = []
    for idx, solucao in enumerate(solucoes):
        print(f"\nAnálise para Época {idx+1}:")
        
        # 🚨 VERIFICAR COLISÕES PRIMEIRO (IGUAL AO evaluate_individual)
        #tem_colisoes = has_collision(solucao, num_uavs, num_timeslots)
        #print(f"🔍 Verificação de colisões: {'❌ TEM COLISÕES' if tem_colisoes else '✅ SEM COLISÕES'}")
        
        # if tem_colisoes:
        #     print(f"🚨 SOLUÇÃO COM COLISÕES DETECTADA!")
        #     print(f"   Fitness = 0.0 (igual ao evaluate_individual)")
        #     print(f"   Não será feita análise detalhada.")
        #     print("="*50)
        #     return [(0.0, "Solução com colisões")]
        
        # Se não há colisões, continuar com a análise normal
        print(f"✅ Solução válida - prosseguindo com análise detalhada...")
        
        # Lista para armazenar todos os C_média_total e C_min_total do epoch
        todos_c_media = []
        todos_c_min = []
        
        # Recriar a solução para cada timeslot
        for t in range(num_timeslots):
            print(f"\nTimeslot {t+1}:")
            positions = []
            angles = []
            
            for i in range(num_uavs):
                idx_pos = t * num_uavs * 2 + i * 2  # MUDANÇA: *2, pois só temos x e y
                x, y = solucao[idx_pos:idx_pos+2]
                positions.append(np.array([x, y]))
                
                # Calcular ângulo dinamicamente
                angle = np.degrees(np.arctan2(jammer_position[1] - y, jammer_position[0] - x))
                angles.append(angle)
            
            # Calcular matriz de comunicação
            comm_matrix, _ = calculate_communication_capacity(angles, [1.0]*num_uavs, positions, jammer_position)
            
            # Gerar métricas detalhadas
            metricas = avaliar_grafo(comm_matrix)
            
            # Exibir resultados
            print("\nMatriz de Comunicação (bps):")
            print(np.round(comm_matrix, 2))
            
            print("\nResumo de Capacidades:")
            print(f"- Capacidades: {metricas['capacidades']}")
            print(f"- Média: {np.mean(metricas['capacidades']):.2f} bps")
            print(f"- Mínima: {min(metricas['capacidades']):.2f} bps")
            print(f"- Máxima: {max(metricas['capacidades']):.2f} bps")
            
            print("\nCaminhos Críticos:")
            for (i,j), data in metricas['caminhos_minimos'].items():
                print(f"UAV {i} → UAV {j}: {data['path']} (Capacidade: {data['capacidade']:.2f} bps)")
            
            print("\nBottlenecks por UAV:")
            for uav, cap in metricas['bottlenecks'].items():
                print(f"UAV {uav}: {cap:.2f} bps")
            
            print(f"\nGrafo é fortemente conexo? {'Sim' if metricas['conectividade'] else 'Não'}")
            
            print("\nLinks Usados:")
            for u, v in metricas['links_usados']:
                print(f"{u} ↔ {v}")
            
            print("\nLinks Não Usados:")
            for u, v in metricas['links_nao_usados']:
                print(f"{u} ↔ {v}")
            
            # Cálculo do fitness usando apenas links usados
            capacidades_usadas = [comm_matrix[u][v] for u, v in metricas['links_usados'] if u < num_uavs and v < num_uavs]
            if capacidades_usadas:
                C_media_total = np.mean(capacidades_usadas)
                C_min_total = min(capacidades_usadas)
            else:
                C_media_total = 0
                C_min_total = 0
            
            # Adicionar às listas do epoch
            todos_c_media.append(C_media_total)
            todos_c_min.append(C_min_total)
            
            print(f"\nCálculo do Fitness para Timeslot {t+1}:")
            print(f"Capacidades Usadas: {capacidades_usadas}")
            print(f"C_média_total = {C_media_total:.2f} bps")
            print(f"C_min_total = {C_min_total:.2f} bps")
            
            resultados.append({
                'epoch': idx+1,
                'timeslot': t+1,
                'metricas': metricas,
                'C_media_total': C_media_total,
                'C_min_total': C_min_total
            })
        
        # ✅ CORREÇÃO: Calcular fitness de cada timeslot e depois fazer a média (IGUAL AO evaluate_individual)
        alpha, beta = 1.0, 1.0
        fitness_por_timeslot = []
        
        for i in range(len(todos_c_media)):
            C_media = todos_c_media[i]
            C_min = todos_c_min[i]
            fitness_timeslot = (C_media ** alpha) * (C_min ** beta)
            fitness_por_timeslot.append(fitness_timeslot)
        
        # Fitness médio do epoch (IGUAL AO evaluate_individual)
        fitness_epoch = np.mean(fitness_por_timeslot) if fitness_por_timeslot else 0
        
        # Calcular também as médias para informação
        media_c_media_epoch = np.mean(todos_c_media) if todos_c_media else 0
        media_c_min_epoch = np.mean(todos_c_min) if todos_c_min else 0
        
        print(f"\n" + "="*50)
        print(f"RESUMO DO EPOCH {idx+1}:")
        print(f"Todos os C_média_total: {[f'{x:.2f}' for x in todos_c_media]}")
        print(f"Todos os C_min_total: {[f'{x:.2f}' for x in todos_c_min]}")
        print(f"Fitness por timeslot: {[f'{x:.4f}' for x in fitness_por_timeslot]}")
        print(f"Média dos C_média_total: {media_c_media_epoch:.2f} bps")
        print(f"Média dos C_min_total: {media_c_min_epoch:.2f} bps")
        print(f"🎯 Fitness do Epoch (CORRETO) = {fitness_epoch:.4f}")
        print(f"✅ Agora deve bater com o valor do gráfico!")
        print("="*50)
    
    return resultados, fitness_epoch, media_c_media_epoch, media_c_min_epoch


In [7]:
resultados, fitness_epoch, media_c_media_epoch, media_c_min_epoch = analisar_comunicacoes_detalhado(
    filename="best_solution_epoch.txt",
    num_uavs=4,  # ou o número correto de UAVs que você está usando
    num_timeslots=6,  # Confirmando que são 6 timeslots
    jammer_position=np.array([0.0,500.0])  # Posição do jammer
)

print(media_c_media_epoch)




Análise para Época 1:
✅ Solução válida - prosseguindo com análise detalhada...

Timeslot 1:

Matriz de Comunicação (bps):
[[   0.    149.61   19.3  4135.57]
 [ 149.61    0.   3279.91   15.7 ]
 [  19.3  3279.91    0.    905.03]
 [4135.57   15.7   905.03    0.  ]]

Resumo de Capacidades:
- Capacidades: [149.6124526800803, 19.29542518891712, 4135.567146522955, 149.6124526800803, 3279.911240858121, 15.695302631263397, 19.29542518891712, 3279.911240858121, 905.0263091151259, 4135.567146522955, 15.695302631263397, 905.0263091151259]
- Média: 1417.52 bps
- Mínima: 15.70 bps
- Máxima: 4135.57 bps

Caminhos Críticos:
UAV 0 → UAV 1: [0, 3, 2, 1] (Capacidade: 905.03 bps)
UAV 0 → UAV 2: [0, 3, 2] (Capacidade: 905.03 bps)
UAV 0 → UAV 3: [0, 3] (Capacidade: 4135.57 bps)
UAV 1 → UAV 0: [1, 2, 3, 0] (Capacidade: 905.03 bps)
UAV 1 → UAV 2: [1, 2] (Capacidade: 3279.91 bps)
UAV 1 → UAV 3: [1, 2, 3] (Capacidade: 905.03 bps)
UAV 2 → UAV 0: [2, 3, 0] (Capacidade: 905.03 bps)
UAV 2 → UAV 1: [2, 1] (Capacida

# dataset permutado para treianr modelo (já vem de trás) - uav_dataset_com_permutacoes.csv

In [ ]:
# já vem de trás igual ao class 3

# criar dataset.csv (valores CONTINUOS) para testar as previsões da interpolação

In [6]:
import pandas as pd
import numpy as np

def gerar_dataset_otimizado(n_jammers=1, m_combinacoes_uav=1):
    """
    Versão otimizada: se ficar preso, recomeça a configuração
    """
    jammers = [(np.random.uniform(0, 120), 500.0) for _ in range(n_jammers)]
    combinacoes_uav = []
    
    while len(combinacoes_uav) < m_combinacoes_uav:
        uavs = []
        tentativas_por_uav = [0, 0, 0, 0]
        
        for i in range(4):
            colocado = False
            
            while not colocado and tentativas_por_uav[i] < 100:
                tentativas_por_uav[i] += 1
                x = np.random.uniform(0, 60)
                y = np.random.uniform(0, 60)
                
                # Verificar distância com os já colocados
                ok = True
                for ux, uy in uavs:
                    if ((x-ux)**2 + (y-uy)**2)**0.5 < 20:
                        ok = False
                        break
                
                if ok:
                    uavs.append((x, y))
                    colocado = True
            
            # Se não conseguiu colocar este UAV, recomeçar tudo
            if not colocado:
                uavs = []
                tentativas_por_uav = [0, 0, 0, 0]
                break
        
        if len(uavs) == 4:
            combinacoes_uav.append(uavs)
    
    # Montar dataset final
    data = []
    for jx, jy in jammers:
        for uavs in combinacoes_uav:
            row = []
            for ux, uy in uavs:
                row.extend([ux, uy])
            row.extend([jx, jy])
            data.append(row)
    
    return pd.DataFrame(data)

# Usar
df = gerar_dataset_otimizado(2, 2)
df.to_csv('dataset.csv', index=False, header=False)
print(f"Criado: {len(df)} linhas")


Criado: 4 linhas


In [10]:
import pandas as pd
import numpy as np

def gerar_dataset_otimizado(n_jammers=1, m_combinacoes_uav=1):
    """
    Versão otimizada: se ficar preso, recomeça a configuração
    """
    # 🎯 INTERPOLAÇÃO DA POSIÇÃO X DO JAMMER
    if n_jammers == 1:
        jammers = [(60.0, 500.0)]  # No meio se só há 1 jammer
    else:
        # Interpolar uniformemente de 0 a 120
        x_positions = np.linspace(0, 120, n_jammers)
        jammers = [(x, 500.0) for x in x_positions]
    
    combinacoes_uav = []
    
    while len(combinacoes_uav) < m_combinacoes_uav:
        uavs = []
        tentativas_por_uav = [0, 0, 0, 0]
        
        for i in range(4):
            colocado = False
            
            while not colocado and tentativas_por_uav[i] < 100:
                tentativas_por_uav[i] += 1
                x = np.random.uniform(0, 60)
                y = np.random.uniform(0, 60)
                
                # Verificar distância com os já colocados
                ok = True
                for ux, uy in uavs:
                    if ((x-ux)**2 + (y-uy)**2)**0.5 < 20:
                        ok = False
                        break
                
                if ok:
                    uavs.append((x, y))
                    colocado = True
            
            # Se não conseguiu colocar este UAV, recomeçar tudo
            if not colocado:
                uavs = []
                tentativas_por_uav = [0, 0, 0, 0]
                break
        
        if len(uavs) == 4:
            combinacoes_uav.append(uavs)
    
    # Montar dataset final
    data = []
    for jx, jy in jammers:
        for uavs in combinacoes_uav:
            row = []
            for ux, uy in uavs:
                row.extend([ux, uy])
            row.extend([jx, jy])
            data.append(row)
    
    return pd.DataFrame(data)

# Usar
df = gerar_dataset_otimizado(10, 10)  # 5 jammers, 2 combinações UAV
df.to_csv('dataset.csv', index=False, header=False)
print(f"Criado: {len(df)} linhas")
print(f"Posições X dos jammers: {df.iloc[:, -2].unique()}")  # Mostrar posições X dos jammers


Criado: 100 linhas
Posições X dos jammers: [  0.          13.33333333  26.66666667  40.          53.33333333
  66.66666667  80.          93.33333333 106.66666667 120.        ]


# inserir soluções CONTINUAS no dataset:csv

In [11]:
import numpy as np
import networkx as nx

def analisar_comunicacoes_detalhado(best_solution, num_uavs, num_timeslots, jammer_position):
    def avaliar_grafo(comm_matrix):
        G = nx.DiGraph()
        
        # 1. Adicionar arestas bidirecionais com capacidades
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j and comm_matrix[i][j] > 0:
                    G.add_edge(i, j, capacity=comm_matrix[i][j])
        
        # 2. Cálculos de métricas
        metricas = {
            'capacidades': [],
            'caminhos_minimos': {},
            'bottlenecks': {},
            'conectividade': None,
            'links_usados': set(),
            'links_nao_usados': set()
        }
        
        # Preencher métricas
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j:
                    # Calcular caminhos mínimos (1/capacity como peso)
                    try:
                        path = nx.shortest_path(G, source=i, target=j, weight=lambda u, v, d: 1/d['capacity'])
                        capacidade_min = min(G[u][v]['capacity'] for u, v in zip(path[:-1], path[1:]))
                        
                        metricas['caminhos_minimos'][(i,j)] = {
                            'path': path,
                            'capacidade': capacidade_min
                        }
                        
                        # Adicionar links usados
                        for u, v in zip(path[:-1], path[1:]):
                            metricas['links_usados'].add((u, v))
                            metricas['links_usados'].add((v, u))  # Adicionar a aresta reversa
                    except:
                        pass
        
        # Calcular bottlenecks para cada nó
        for node in G.nodes():
            metricas['bottlenecks'][node] = min(
                [d['capacity'] for _, _, d in G.edges(node, data=True)],
                default=0
            )
        
        # Verificar conectividade
        metricas['conectividade'] = nx.is_strongly_connected(G)
        
        # Coletar todas as capacidades
        metricas['capacidades'] = [d['capacity'] for _, _, d in G.edges(data=True)]
        
        # Identificar links não usados
        all_links = {(i, j) for i in range(num_uavs) for j in range(num_uavs) if i != j}
        metricas['links_nao_usados'] = all_links - metricas['links_usados']
        
        return metricas

    # A melhor solução é passada diretamente
    solucoes = [best_solution]  # Colocar a melhor solução em uma lista para iteração

    resultados = []
    for idx, solucao in enumerate(solucoes):
        #print(f"\nAnálise para Época {idx+1}:")
        
        # 🚨 VERIFICAR COLISÕES PRIMEIRO (IGUAL AO evaluate_individual)
        #tem_colisoes = has_collision(solucao, num_uavs, num_timeslots)
        #print(f"🔍 Verificação de colisões: {'❌ TEM COLISÕES' if tem_colisoes else '✅ SEM COLISÕES'}")
        
        # if tem_colisoes:
        #     # print(f"🚨 SOLUÇÃO COM COLISÕES DETECTADA!")
        #     # print(f"   Fitness = 0.0 (igual ao evaluate_individual)")
        #     # print(f"   Não será feita análise detalhada.")
        #     # print("="*50)
        #     # return [(0.0, "Solução com colisões")]
        
        # Se não há colisões, continuar com a análise normal
        #print(f"✅ Solução válida - prosseguindo com análise detalhada...")
        
        # Lista para armazenar todos os C_média_total e C_min_total do epoch
        todos_c_media = []
        todos_c_min = []
        
        # Recriar a solução para cada timeslot
        for t in range(num_timeslots):
            # print(f"\nTimeslot {t+1}:")
            positions = []
            angles = []
            
            for i in range(num_uavs):
                idx_pos = t * num_uavs * 2 + i * 2  # MUDANÇA: *2, pois só temos x e y
                x, y = solucao[idx_pos:idx_pos+2]
                positions.append(np.array([x, y]))
                
                # Calcular ângulo dinamicamente
                angle = np.degrees(np.arctan2(jammer_position[1] - y, jammer_position[0] - x))
                angles.append(angle)
            
            # Calcular matriz de comunicação
            comm_matrix, _ = calculate_communication_capacity(angles, [1.0]*num_uavs, positions, jammer_position)
            
            # Gerar métricas detalhadas
            metricas = avaliar_grafo(comm_matrix)
            
            # # Exibir resultados
            # print("\nMatriz de Comunicação (bps):")
            # print(np.round(comm_matrix, 2))
            
            # print("\nResumo de Capacidades:")
            # print(f"- Capacidades: {metricas['capacidades']}")
            # print(f"- Média: {np.mean(metricas['capacidades']):.2f} bps")
            # print(f"- Mínima: {min(metricas['capacidades']):.2f} bps")
            # print(f"- Máxima: {max(metricas['capacidades']):.2f} bps")
            
            # print("\nCaminhos Críticos:")
            # for (i,j), data in metricas['caminhos_minimos'].items():
            #     print(f"UAV {i} → UAV {j}: {data['path']} (Capacidade: {data['capacidade']:.2f} bps)")
            
            # print("\nBottlenecks por UAV:")
            # for uav, cap in metricas['bottlenecks'].items():
            #     print(f"UAV {uav}: {cap:.2f} bps")
            
            # print(f"\nGrafo é fortemente conexo? {'Sim' if metricas['conectividade'] else 'Não'}")
            
            # print("\nLinks Usados:")
            # for u, v in metricas['links_usados']:
            #     print(f"{u} ↔ {v}")
            
            # print("\nLinks Não Usados:")
            # for u, v in metricas['links_nao_usados']:
            #     print(f"{u} ↔ {v}")
            
            # Cálculo do fitness usando apenas links usados
            capacidades_usadas = [comm_matrix[u][v] for u, v in metricas['links_usados'] if u < num_uavs and v < num_uavs]
            if capacidades_usadas:
                C_media_total = np.mean(capacidades_usadas)
                C_min_total = min(capacidades_usadas)
            else:
                C_media_total = 0
                C_min_total = 0
            
            # Adicionar às listas do epoch
            todos_c_media.append(C_media_total)
            todos_c_min.append(C_min_total)
            
            # print(f"\nCálculo do Fitness para Timeslot {t+1}:")
            # print(f"Capacidades Usadas: {capacidades_usadas}")
            # print(f"C_média_total = {C_media_total:.2f} bps")
            # print(f"C_min_total = {C_min_total:.2f} bps")
            
            resultados.append({
                'epoch': idx+1,
                'timeslot': t+1,
                'metricas': metricas,
                'C_media_total': C_media_total,
                'C_min_total': C_min_total
            })
        
        # ✅ CORREÇÃO: Calcular fitness de cada timeslot e depois fazer a média (IGUAL AO evaluate_individual)
        alpha, beta = 1.0, 1.0
        fitness_por_timeslot = []
        
        for i in range(len(todos_c_media)):
            C_media = todos_c_media[i]
            C_min = todos_c_min[i]
            fitness_timeslot = (C_media ** alpha) * (C_min ** beta)
            fitness_por_timeslot.append(fitness_timeslot)
        
        # Fitness médio do epoch (IGUAL AO evaluate_individual)
        fitness_epoch = np.mean(fitness_por_timeslot) if fitness_por_timeslot else 0
        
        # Calcular também as médias para informação
        media_c_media_epoch = np.mean(todos_c_media) if todos_c_media else 0
        media_c_min_epoch = np.mean(todos_c_min) if todos_c_min else 0
        
        # print(f"\n" + "="*50)
        # print(f"RESUMO DO EPOCH {idx+1}:")
        # print(f"Todos os C_média_total: {[f'{x:.2f}' for x in todos_c_media]}")
        # print(f"Todos os C_min_total: {[f'{x:.2f}' for x in todos_c_min]}")
        # print(f"Fitness por timeslot: {[f'{x:.4f}' for x in fitness_por_timeslot]}")
        # print(f"Média dos C_média_total: {media_c_media_epoch:.2f} bps")
        # print(f"Média dos C_min_total: {media_c_min_epoch:.2f} bps")
        #print(f"🎯 Fitness do Epoch (CORRETO) = {fitness_epoch:.4f}")
        # print(f"✅ Agora deve bater com o valor do gráfico!")
        # print("="*50)
    
    return resultados, fitness_epoch, media_c_media_epoch, media_c_min_epoch


In [12]:
import pandas as pd
import numpy as np

# Função para carregar combinações de UAVs e jammers de um arquivo CSV
def load_combinations_from_csv(filename):
    return pd.read_csv(filename, header=None)

# Função para salvar as posições iniciais, a posição do jammer e as posições finais em um novo dataset
def save_combined_positions(original_data, final_positions, fitness_values, index, filename="uav_jammer_combined.csv"):
    # Criar o nome do arquivo combinado
    combined_filename = filename.replace('.csv', '_combined.csv')

    # Extrair as posições iniciais dos UAVs
    swarm = original_data.iloc[index, :-2].values.reshape(-1, 2)  # Posições dos UAVs
    jammer_position = original_data.iloc[index, -2:].values  # Posição do jammer

    # Extrair as posições finais
    final_xs = [pos[0] for pos in final_positions]  # Coordenadas x
    final_ys = [pos[1] for pos in final_positions]  # Coordenadas y

    # Criar um dicionário para a linha a ser salva
    combined_row = {
        'Jammer_X': jammer_position[0],
        'Jammer_Y': jammer_position[1],
        'Fitness': fitness_values[0],
        'Média_C_media_total': fitness_values[1],
        'Média_C_min_total': fitness_values[2]
    }
    
    # Adicionar posições iniciais
    for uav in range(len(swarm)):
        combined_row[f'Initial_X{uav + 1}'] = swarm[uav][0]
        combined_row[f'Initial_Y{uav + 1}'] = swarm[uav][1]

    # Adicionar posições finais
    for uav in range(len(final_xs)):
        combined_row[f'Final_X{uav + 1}'] = final_xs[uav]
        combined_row[f'Final_Y{uav + 1}'] = final_ys[uav]

    # Criar um DataFrame a partir do dicionário
    combined_df = pd.DataFrame([combined_row])

    # Salvar o DataFrame em um arquivo CSV, adicionando ao final
    combined_df.to_csv(combined_filename, mode='a', header=not pd.io.common.file_exists(combined_filename), index=False)
    #print(f'Dataset salvo para a iteração {index + 1} no arquivo: {combined_filename}')

# Exemplo de uso
dataset_filename = 'dataset.csv'  # Substitua pelo nome do seu arquivo
original_data = load_combinations_from_csv(dataset_filename)

# Lista para armazenar as posições finais otimizadas
final_positions = []
fitness_values = []  # Lista para armazenar os valores de fitness e médias

# Iterar sobre cada linha do dataset
for epoch, (index, row) in enumerate(original_data.iterrows()):
    # Extrair as posições dos UAVs da linha
    swarm = []
    for i in range(0, len(row) - 2, 2):  # Ignorar as últimas duas colunas (posições do jammer)
        x = row.iloc[i]
        y = row.iloc[i + 1]
        swarm.append((x, y))
    
    # Posição do jammer (últimas duas colunas)
    jammer_position = (row.iloc[-2], row.iloc[-1])  # Usar .iloc para acessar as últimas colunas
    
    # Executar a simulação
    best_solution, initial_positions, jammer_position, all_min_capacities, all_interference_matrices, avg_min_capacity = simulate_uavs_with_ga(
        num_epochs=1, 
        num_timeslots=6, 
        timeslot_length=60, 
        num_uavs=4,
        population_size=50,
        generations=50,
        manual_initial_positions=swarm,
        jammer_position=jammer_position,
        min_y=0.0,
        max_y=60.0,
        epoch_total_length=120
    )

    # Analisar a melhor solução
    resultados, fitness_epoch, media_c_media_epoch, media_c_min_epoch = analisar_comunicacoes_detalhado(
        best_solution=best_solution,  # Passar a melhor solução
        num_uavs=4,  # ou o número correto de UAVs que você está usando
        num_timeslots=6,  # Confirmando que são 6 timeslots
        jammer_position=np.array(jammer_position)  # Posição do jammer
    )

    # Extrair as posições finais
    num_timeslots = 6
    num_uavs = 4  # Número de UAVs
    final_positions_for_epoch = []

    # Calcular o índice inicial para as posições finais do último timeslot
    final_positions_start_index = (num_timeslots - 1) * num_uavs * 2  # Posições do último timeslot

    # Iterar sobre cada UAV para extrair suas posições finais
    for uav in range(num_uavs):
        final_x = best_solution[final_positions_start_index + uav * 2]     # Coordenada x do UAV
        final_y = best_solution[final_positions_start_index + uav * 2 + 1] # Coordenada y do UAV
        final_positions_for_epoch.append((final_x, final_y))  # Adicionar a posição final do UAV

    # Adicionar as posições finais do epoch à lista principal
    final_positions.append(final_positions_for_epoch)

    # Adicionar os valores de fitness e médias à lista
    fitness_values.append((fitness_epoch, media_c_media_epoch, media_c_min_epoch))

    # Salvar as posições iniciais, a posição do jammer e as posições finais no novo dataset
    save_combined_positions(original_data, final_positions_for_epoch, 
                             (fitness_epoch, media_c_media_epoch, media_c_min_epoch), 
                             index, dataset_filename)


# permutar

In [13]:
import pandas as pd
import itertools
from tqdm import tqdm

def generate_uav_permutations(csv_file, output_file):
    """
    Gera todas as permutações dos UAVs para cada linha do dataset
    Mantém as trajetórias (inicial→final) ligadas
    """
    
    print("📂 Carregando dataset...")
    df = pd.read_csv(csv_file)
    print(f"✅ {len(df)} linhas carregadas")
    
    # Lista para armazenar todas as novas linhas
    new_rows = []
    
    # Gerar todas as permutações possíveis dos UAVs (0,1,2,3)
    uav_indices = [0, 1, 2, 3]
    all_permutations = list(itertools.permutations(uav_indices))
    
    # Remover a permutação original (0,1,2,3)
    original_permutation = (0, 1, 2, 3)
    permutations_to_use = [p for p in all_permutations if p != original_permutation]
    
    print(f"🔄 Gerando {len(permutations_to_use)} permutações para cada linha...")
    print(f"📊 Total de novas linhas: {len(df)} × {len(permutations_to_use)} = {len(df) * len(permutations_to_use)}")
    
    # Para cada linha original
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processando linhas"):
        
        # Extrair trajetórias originais (inicial→final) para cada UAV
        trajectories = []
        for uav in range(4):
            initial_x = row[f'Initial_X{uav+1}']
            initial_y = row[f'Initial_Y{uav+1}']
            final_x = row[f'Final_X{uav+1}']
            final_y = row[f'Final_Y{uav+1}']
            
            trajectories.append({
                'initial_x': initial_x,
                'initial_y': initial_y,
                'final_x': final_x,
                'final_y': final_y
            })
        
        # Para cada permutação (exceto a original)
        for perm in permutations_to_use:
            # Criar nova linha
            new_row = row.copy()
            
            # Aplicar permutação: UAV i recebe a trajetória do UAV perm[i]
            for uav_new in range(4):
                uav_original = perm[uav_new]
                
                # UAV (uav_new+1) recebe a trajetória do UAV (uav_original+1)
                new_row[f'Initial_X{uav_new+1}'] = trajectories[uav_original]['initial_x']
                new_row[f'Initial_Y{uav_new+1}'] = trajectories[uav_original]['initial_y']
                new_row[f'Final_X{uav_new+1}'] = trajectories[uav_original]['final_x']
                new_row[f'Final_Y{uav_new+1}'] = trajectories[uav_original]['final_y']
            
            # Adicionar à lista
            new_rows.append(new_row)
    
    # Criar DataFrame com todas as linhas (originais + permutações)
    print("📝 Criando DataFrame final...")
    
    # CORREÇÃO: Concatenar DataFrames corretamente
    new_df = pd.DataFrame(new_rows)
    final_df = pd.concat([df, new_df], ignore_index=True)
    
    # Salvar
    print(f"💾 Salvando {len(final_df)} linhas em {output_file}...")
    final_df.to_csv(output_file, index=False)
    
    print("✅ Concluído!")
    print(f"📊 Estatísticas:")
    print(f"   Linhas originais: {len(df)}")
    print(f"   Novas linhas: {len(new_rows)}")
    print(f"   Total final: {len(final_df)}")
    print(f"   Fator de multiplicação: {len(final_df) / len(df):.1f}x")
    
    return final_df

# ================================
# EXEMPLO DE USO
# ================================uav_dataset_reference_2

# Gerar permutações
input_file = "dataset_combined.csv"
output_file = "dataset_combined_com_permutacoes.csv"

df_expandido = generate_uav_permutations(input_file, output_file)

# Verificar algumas linhas
print("\n🔍 Primeiras 3 linhas do dataset expandido:")
print(df_expandido[['Initial_X1', 'Initial_Y1', 'Initial_X2', 'Initial_Y2', 
                   'Final_X1', 'Final_Y1', 'Final_X2', 'Final_Y2']].head(3))

📂 Carregando dataset...
✅ 100 linhas carregadas
🔄 Gerando 23 permutações para cada linha...
📊 Total de novas linhas: 100 × 23 = 2300


Processando linhas: 100%|██████████| 100/100 [00:00<00:00, 128.01it/s]


📝 Criando DataFrame final...
💾 Salvando 2400 linhas em dataset_combined_com_permutacoes.csv...
✅ Concluído!
📊 Estatísticas:
   Linhas originais: 100
   Novas linhas: 2300
   Total final: 2400
   Fator de multiplicação: 24.0x

🔍 Primeiras 3 linhas do dataset expandido:
   Initial_X1  Initial_Y1  Initial_X2  Initial_Y2    Final_X1   Final_Y1  \
0   12.185706    9.871361   38.189219   21.308982   90.661961  50.014225   
1   35.826275   53.588470   43.208197    7.507231  112.700233  52.452538   
2   54.547927   52.366718   26.083593   55.735149  111.395031  54.164282   

     Final_X2   Final_Y2  
0  110.436123  23.967412  
1  103.052855  31.774169  
2   87.566409  49.356443  


# KNN

In [7]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsRegressor

def create_predictions_dataset(train_file, test_file, output_file, n_neighbors=4):
    """
    Treina KNN com um dataset base e faz predições em outro dataset
    
    Args:
        train_file: Dataset para treinar o KNN (base)
        test_file: Dataset para fazer predições
        output_file: Arquivo de saída com as predições
        n_neighbors: Número de vizinhos para o KNN
    """
    
    # 1. Carregar dataset de treino (BASE)
    print("Carregando dataset de treino...")
    train_df = pd.read_csv(train_file)
    print(f"Dataset de treino carregado com {len(train_df)} linhas")
    
    # 2. Carregar dataset de teste
    print("Carregando dataset de teste...")
    test_df = pd.read_csv(test_file)
    print(f"Dataset de teste carregado com {len(test_df)} linhas")
    
    # 3. Verificar se temos amostras suficientes para treino
    if len(train_df) < n_neighbors:
        print(f"AVISO: Dataset de treino tem apenas {len(train_df)} linhas, reduzindo n_neighbors para {len(train_df)}")
        n_neighbors = len(train_df)
    
    # 4. Preparar dados de treino
    feature_columns = ['Jammer_X', 'Jammer_Y', 
                      'Initial_X1', 'Initial_Y1', 'Initial_X2', 'Initial_Y2', 
                      'Initial_X3', 'Initial_Y3', 'Initial_X4', 'Initial_Y4']
    
    target_columns = ['Final_X1', 'Final_Y1', 'Final_X2', 'Final_Y2',
                     'Final_X3', 'Final_Y3', 'Final_X4', 'Final_Y4']
    
    # Dados de treino
    X_train = train_df[feature_columns].values
    y_train = train_df[target_columns].values
    
    # 5. Treinar o modelo KNN
    print("Treinando modelo KNN...")
    knn = KNeighborsRegressor(n_neighbors=n_neighbors, weights='distance')
    knn.fit(X_train, y_train)
    print("Modelo KNN treinado com sucesso!")
    
    # 6. Preparar dados de teste
    X_test = test_df[feature_columns].values
    
    # 7. Fazer predições para todo o dataset de teste
    print("Fazendo predições...")
    all_predictions = knn.predict(X_test)
    
    # 8. Criar lista para armazenar os novos dados
    new_data = []
    
    # 9. Processar cada linha do dataset de teste
    for index, row in test_df.iterrows():
        # Predições para esta linha
        predicted_final_positions = all_predictions[index]
        
        # Extrair dados da linha atual
        jammer_x = row['Jammer_X']
        jammer_y = row['Jammer_Y']
        
        # Posições iniciais
        initial_positions = [
            row['Initial_X1'], row['Initial_Y1'],
            row['Initial_X2'], row['Initial_Y2'], 
            row['Initial_X3'], row['Initial_Y3'],
            row['Initial_X4'], row['Initial_Y4']
        ]
        
        # Posições finais reais
        real_final_positions = [
            row['Final_X1'], row['Final_Y1'],
            row['Final_X2'], row['Final_Y2'],
            row['Final_X3'], row['Final_Y3'], 
            row['Final_X4'], row['Final_Y4']
        ]
        
        # Criar nova linha de dados
        new_row = {
            'Jammer_X': jammer_x,
            'Jammer_Y': jammer_y,
            'Initial_X1': row['Initial_X1'],
            'Initial_Y1': row['Initial_Y1'],
            'Initial_X2': row['Initial_X2'],
            'Initial_Y2': row['Initial_Y2'],
            'Initial_X3': row['Initial_X3'],
            'Initial_Y3': row['Initial_Y3'],
            'Initial_X4': row['Initial_X4'],
            'Initial_Y4': row['Initial_Y4'],
            'Predicted_Final_X1': predicted_final_positions[0],
            'Predicted_Final_Y1': predicted_final_positions[1],
            'Predicted_Final_X2': predicted_final_positions[2],
            'Predicted_Final_Y2': predicted_final_positions[3],
            'Predicted_Final_X3': predicted_final_positions[4],
            'Predicted_Final_Y3': predicted_final_positions[5],
            'Predicted_Final_X4': predicted_final_positions[6],
            'Predicted_Final_Y4': predicted_final_positions[7],
            'Real_Final_X1': real_final_positions[0],
            'Real_Final_Y1': real_final_positions[1],
            'Real_Final_X2': real_final_positions[2],
            'Real_Final_Y2': real_final_positions[3],
            'Real_Final_X3': real_final_positions[4],
            'Real_Final_Y3': real_final_positions[5],
            'Real_Final_X4': real_final_positions[6],
            'Real_Final_Y4': real_final_positions[7]
        }
        
        new_data.append(new_row)
        
        # Mostrar progresso a cada 100 linhas
        if (index + 1) % 100 == 0:
            print(f"Processadas {index + 1} linhas...")
    
    # 10. Criar DataFrame com os novos dados
    new_df = pd.DataFrame(new_data)
    
    # 11. Salvar o novo dataset
    new_df.to_csv(output_file, index=False)
    print(f"Novo dataset salvo em '{output_file}' com {len(new_df)} linhas")
    
    # 12. Mostrar algumas estatísticas
    print("\n=== ESTATÍSTICAS DE ERRO ===")
    
    # Calcular erros médios para cada UAV
    for uav in range(1, 5):
        pred_x_col = f'Predicted_Final_X{uav}'
        pred_y_col = f'Predicted_Final_Y{uav}'
        real_x_col = f'Real_Final_X{uav}'
        real_y_col = f'Real_Final_Y{uav}'
        
        # Erro em X
        error_x = np.mean(np.abs(new_df[pred_x_col] - new_df[real_x_col]))
        # Erro em Y  
        error_y = np.mean(np.abs(new_df[pred_y_col] - new_df[real_y_col]))
        # Erro euclidiano
        error_euclidean = np.mean(np.sqrt((new_df[pred_x_col] - new_df[real_x_col])**2 + 
                                         (new_df[pred_y_col] - new_df[real_y_col])**2))
        
        print(f"UAV {uav}:")
        print(f"  Erro médio absoluto X: {error_x:.2f}")
        print(f"  Erro médio absoluto Y: {error_y:.2f}")
        print(f"  Erro euclidiano médio: {error_euclidean:.2f}")
    
    # Erro geral
    all_errors_euclidean = []
    for uav in range(1, 5):
        pred_x_col = f'Predicted_Final_X{uav}'
        pred_y_col = f'Predicted_Final_Y{uav}'
        real_x_col = f'Real_Final_X{uav}'
        real_y_col = f'Real_Final_Y{uav}'
        
        errors = np.sqrt((new_df[pred_x_col] - new_df[real_x_col])**2 + 
                        (new_df[pred_y_col] - new_df[real_y_col])**2)
        all_errors_euclidean.extend(errors)
    
    print(f"\n=== ERRO GERAL ===")
    print(f"Erro euclidiano médio geral: {np.mean(all_errors_euclidean):.2f}")
    print(f"Desvio padrão do erro: {np.std(all_errors_euclidean):.2f}")
    print(f"Erro máximo: {np.max(all_errors_euclidean):.2f}")
    print(f"Erro mínimo: {np.min(all_errors_euclidean):.2f}")
    
    return new_df

# Executar a função
if __name__ == "__main__":
    # Criar o novo dataset com predições
    new_dataset = create_predictions_dataset(
        train_file="uav_dataset_reference_1_com_permutacoes.csv",        # Dataset para treinar o KNN
        test_file="dataset_combined_com_permutacoes.csv",        # Dataset para fazer predições
        output_file="inter_results/11n_knn.csv",   # Dataset resultado com predições
        n_neighbors=11
    )
    
    # Mostrar as primeiras linhas do novo dataset
    print("\n=== PRIMEIRAS 5 LINHAS DO DATASET RESULTADO ===")
    print(new_dataset.head())


Carregando dataset de treino...
Dataset de treino carregado com 305760 linhas
Carregando dataset de teste...
Dataset de teste carregado com 2400 linhas
Treinando modelo KNN...
Modelo KNN treinado com sucesso!
Fazendo predições...
Processadas 100 linhas...
Processadas 200 linhas...
Processadas 300 linhas...
Processadas 400 linhas...
Processadas 500 linhas...
Processadas 600 linhas...
Processadas 700 linhas...
Processadas 800 linhas...
Processadas 900 linhas...
Processadas 1000 linhas...
Processadas 1100 linhas...
Processadas 1200 linhas...
Processadas 1300 linhas...
Processadas 1400 linhas...
Processadas 1500 linhas...
Processadas 1600 linhas...
Processadas 1700 linhas...
Processadas 1800 linhas...
Processadas 1900 linhas...
Processadas 2000 linhas...
Processadas 2100 linhas...
Processadas 2200 linhas...
Processadas 2300 linhas...
Processadas 2400 linhas...
Novo dataset salvo em 'inter_results/11n_knn.csv' com 2400 linhas

=== ESTATÍSTICAS DE ERRO ===
UAV 1:
  Erro médio absoluto X: 11.

In [2]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler  # ← ADICIONAR
import pickle
import os

def train_and_save_knn(train_file, model_file, scaler_file, n_neighbors=4):
    """
    Treina KNN e salva o modelo + scaler em .pkl
    """
    
    # Criar diretório se não existir
    os.makedirs(os.path.dirname(model_file), exist_ok=True)
    
    # 1. Carregar dataset de treino
    print("Carregando dataset de treino...")
    train_df = pd.read_csv(train_file)
    print(f"Dataset de treino carregado com {len(train_df)} linhas")
    
    # 2. Verificar se temos amostras suficientes
    if len(train_df) < n_neighbors:
        print(f"AVISO: Dataset de treino tem apenas {len(train_df)} linhas, reduzindo n_neighbors para {len(train_df)}")
        n_neighbors = len(train_df)
    
    # 3. Preparar dados
    feature_columns = ['Jammer_X', 'Jammer_Y', 
                      'Initial_X1', 'Initial_Y1', 'Initial_X2', 'Initial_Y2', 
                      'Initial_X3', 'Initial_Y3', 'Initial_X4', 'Initial_Y4']
    
    target_columns = ['Final_X1', 'Final_Y1', 'Final_X2', 'Final_Y2',
                     'Final_X3', 'Final_Y3', 'Final_X4', 'Final_Y4']
    
    X_train = train_df[feature_columns].values
    y_train = train_df[target_columns].values
    
    # 4. 🆕 APLICAR SCALER
    print("Aplicando StandardScaler...")
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    
    # 5. Treinar modelo
    print("Treinando modelo KNN...")
    knn = KNeighborsRegressor(n_neighbors=n_neighbors, weights='distance')
    knn.fit(X_train_scaled, y_train)  # ← Usar dados escalados
    print("Modelo KNN treinado com sucesso!")
    
    # 6. 🆕 SALVAR MODELO E SCALER
    with open(model_file, 'wb') as f:
        pickle.dump(knn, f)
    print(f"Modelo salvo em {model_file}")
    
    with open(scaler_file, 'wb') as f:
        pickle.dump(scaler, f)
    print(f"Scaler salvo em {scaler_file}")
    
    return knn, scaler

def load_and_predict_knn(model_file, scaler_file, test_file, output_file):
    """
    Carrega modelo + scaler KNN e faz predições
    """
    
    # 1. 🆕 CARREGAR MODELO E SCALER
    with open(model_file, 'rb') as f:
        knn = pickle.load(f)
    print(f"Modelo carregado de {model_file}")
    
    with open(scaler_file, 'rb') as f:
        scaler = pickle.load(f)
    print(f"Scaler carregado de {scaler_file}")
    
    # 2. Carregar dataset de teste
    print("Carregando dataset de teste...")
    test_df = pd.read_csv(test_file)
    print(f"Dataset de teste carregado com {len(test_df)} linhas")
    
    # 3. Preparar dados de teste
    feature_columns = ['Jammer_X', 'Jammer_Y', 
                      'Initial_X1', 'Initial_Y1', 'Initial_X2', 'Initial_Y2', 
                      'Initial_X3', 'Initial_Y3', 'Initial_X4', 'Initial_Y4']
    
    X_test = test_df[feature_columns].values
    
    # 4. 🆕 APLICAR SCALER NOS DADOS DE TESTE
    X_test_scaled = scaler.transform(X_test)  # ← Usar transform (não fit_transform)
    
    # 5. Fazer predições
    print("Fazendo predições...")
    all_predictions = knn.predict(X_test_scaled)  # ← Usar dados escalados
    
    # 6. Resto do código igual...
    new_data = []
    
    for index, row in test_df.iterrows():
        predicted_final_positions = all_predictions[index]
        
        new_row = {
            'Jammer_X': row['Jammer_X'],
            'Jammer_Y': row['Jammer_Y'],
            'Initial_X1': row['Initial_X1'],
            'Initial_Y1': row['Initial_Y1'],
            'Initial_X2': row['Initial_X2'],
            'Initial_Y2': row['Initial_Y2'],
            'Initial_X3': row['Initial_X3'],
            'Initial_Y3': row['Initial_Y3'],
            'Initial_X4': row['Initial_X4'],
            'Initial_Y4': row['Initial_Y4'],
            'Predicted_Final_X1': predicted_final_positions[0],
            'Predicted_Final_Y1': predicted_final_positions[1],
            'Predicted_Final_X2': predicted_final_positions[2],
            'Predicted_Final_Y2': predicted_final_positions[3],
            'Predicted_Final_X3': predicted_final_positions[4],
            'Predicted_Final_Y3': predicted_final_positions[5],
            'Predicted_Final_X4': predicted_final_positions[6],
            'Predicted_Final_Y4': predicted_final_positions[7],
            'Real_Final_X1': row['Final_X1'],
            'Real_Final_Y1': row['Final_Y1'],
            'Real_Final_X2': row['Final_X2'],
            'Real_Final_Y2': row['Final_Y2'],
            'Real_Final_X3': row['Final_X3'],
            'Real_Final_Y3': row['Final_Y3'],
            'Real_Final_X4': row['Final_X4'],
            'Real_Final_Y4': row['Final_Y4']
        }
        
        new_data.append(new_row)
        
        if (index + 1) % 100 == 0:
            print(f"Processadas {index + 1} linhas...")
    
    # 7. Salvar resultado
    new_df = pd.DataFrame(new_data)
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    new_df.to_csv(output_file, index=False)
    print(f"Novo dataset salvo em '{output_file}' com {len(new_df)} linhas")
    
    # 8. Estatísticas (código igual...)
    print("\n=== ESTATÍSTICAS DE ERRO ===")
    
    for uav in range(1, 5):
        pred_x_col = f'Predicted_Final_X{uav}'
        pred_y_col = f'Predicted_Final_Y{uav}'
        real_x_col = f'Real_Final_X{uav}'
        real_y_col = f'Real_Final_Y{uav}'
        
        error_x = np.mean(np.abs(new_df[pred_x_col] - new_df[real_x_col]))
        error_y = np.mean(np.abs(new_df[pred_y_col] - new_df[real_y_col]))
        error_euclidean = np.mean(np.sqrt((new_df[pred_x_col] - new_df[real_x_col])**2 + 
                                         (new_df[pred_y_col] - new_df[real_y_col])**2))
        
        print(f"UAV {uav}:")
        print(f"  Erro médio absoluto X: {error_x:.2f}")
        print(f"  Erro médio absoluto Y: {error_y:.2f}")
        print(f"  Erro euclidiano médio: {error_euclidean:.2f}")
    
    return new_df

def create_predictions_dataset_with_pkl(train_file, test_file, output_file, model_file, scaler_file, n_neighbors=4):
    """
    Função completa que treina, salva, carrega e faz predições
    """
    
    # 1. Treinar e salvar modelo + scaler
    print("=== FASE 1: TREINAR E SALVAR MODELO + SCALER ===")
    train_and_save_knn(train_file, model_file, scaler_file, n_neighbors)
    
    # 2. Carregar modelo + scaler e fazer predições
    print("\n=== FASE 2: CARREGAR MODELO + SCALER E FAZER PREDIÇÕES ===")
    new_df = load_and_predict_knn(model_file, scaler_file, test_file, output_file)
    
    return new_df

# Executar para 2 vizinhos
if __name__ == "__main__":

    
# Executar de 2 a 10 vizinhos
    for n in range(1, 16):
        print(f"\n\n\n🔹🔹🔹 INICIANDO PARA {n} VIZINHOS 🔹🔹🔹\n")
        new_dataset = create_predictions_dataset_with_pkl(
            train_file="uav_dataset_reference_1_com_permutacoes.csv",
            test_file="dataset_combined_com_permutacoes.csv",
            output_file=f"inter_results/{n}n_knn.csv",
            model_file=f"models/knn_{n}n_model.pkl",
            scaler_file=f"models/knn_{n}n_scaler.pkl",  # ← NOVO: ficheiro do scaler
            n_neighbors=n
        )





🔹🔹🔹 INICIANDO PARA 1 VIZINHOS 🔹🔹🔹

=== FASE 1: TREINAR E SALVAR MODELO + SCALER ===
Carregando dataset de treino...
Dataset de treino carregado com 305760 linhas
Aplicando StandardScaler...
Treinando modelo KNN...
Modelo KNN treinado com sucesso!
Modelo salvo em models/knn_1n_model.pkl
Scaler salvo em models/knn_1n_scaler.pkl

=== FASE 2: CARREGAR MODELO + SCALER E FAZER PREDIÇÕES ===
Modelo carregado de models/knn_1n_model.pkl
Scaler carregado de models/knn_1n_scaler.pkl
Carregando dataset de teste...
Dataset de teste carregado com 2400 linhas
Fazendo predições...
Processadas 100 linhas...
Processadas 200 linhas...
Processadas 300 linhas...
Processadas 400 linhas...
Processadas 500 linhas...
Processadas 600 linhas...
Processadas 700 linhas...
Processadas 800 linhas...
Processadas 900 linhas...
Processadas 1000 linhas...
Processadas 1100 linhas...
Processadas 1200 linhas...
Processadas 1300 linhas...
Processadas 1400 linhas...
Processadas 1500 linhas...
Processadas 1600 linhas...
Pr

# random dataset

In [3]:
import pandas as pd
import numpy as np

def criar_dataset_aleatorio_continuo_sem_restricoes(caminho_entrada, caminho_saida):
    """
    Lê o CSV original e cria uma nova versão com posições finais TOTALMENTE aleatórias
    SEM verificar distância mínima - qualquer posição é válida
    
    Args:
        caminho_entrada: Caminho do CSV original
        caminho_saida: Caminho onde salvar o novo CSV com valores aleatórios
    """
    
    print("📂 Carregando dataset original...")
    df = pd.read_csv(caminho_entrada)
    print(f"✅ Dataset carregado com {len(df)} linhas")
    
    # Valores contínuos para X e Y (área alvo)
    x_min, x_max = 60, 120  # Área alvo X
    y_min, y_max = 0, 60    # Área alvo Y
    
    print("🎲 Gerando valores TOTALMENTE aleatórios (sem restrições de distância)...")
    
    # Gerar valores aleatórios para cada linha
    for index in range(len(df)):
        if index % 1000 == 0:  # Progress indicator
            print(f"Processando linha {index}/{len(df)}")
        
        # Gerar posições TOTALMENTE aleatórias para os 4 UAVs
        for uav in range(1, 5):
            x = np.random.uniform(x_min, x_max)  # ← Qualquer X
            y = np.random.uniform(y_min, y_max)  # ← Qualquer Y
            
            df.loc[index, f'Predicted_Final_X{uav}'] = x
            df.loc[index, f'Predicted_Final_Y{uav}'] = y
    
    # Salvar o novo dataset
    print(f"💾 Salvando dataset aleatório em: {caminho_saida}")
    df.to_csv(caminho_saida, index=False)
    
    print(f"✅ Dataset aleatório criado com sucesso!")
    print(f"📊 {len(df)} linhas processadas")
    
    # Mostrar estatísticas dos valores gerados
    print(f"\n📈 Estatísticas dos valores aleatórios:")
    for uav in range(1, 5):
        x_col = f'Predicted_Final_X{uav}'
        y_col = f'Predicted_Final_Y{uav}'
        
        x_min_val = df[x_col].min()
        x_max_val = df[x_col].max()
        y_min_val = df[y_col].min()
        y_max_val = df[y_col].max()
        
        print(f"  UAV {uav}: X=[{x_min_val:.2f}, {x_max_val:.2f}], Y=[{y_min_val:.2f}, {y_max_val:.2f}]")
    
    # Verificar algumas distâncias (só para informação)
    print(f"\n🔍 Distâncias entre UAVs (primeiras 3 linhas - só informativo):")
    for i in range(min(3, len(df))):
        posicoes = []
        for uav in range(1, 5):
            x = df.loc[i, f'Predicted_Final_X{uav}']
            y = df.loc[i, f'Predicted_Final_Y{uav}']
            posicoes.append((x, y))
        
        # Calcular distâncias (só para mostrar)
        distancias = []
        for j in range(4):
            for k in range(j + 1, 4):
                dist = np.sqrt((posicoes[j][0] - posicoes[k][0])**2 + 
                              (posicoes[j][1] - posicoes[k][1])**2)
                distancias.append(dist)
        
        dist_min = min(distancias)
        print(f"  Linha {i}: Distância mínima = {dist_min:.2f}m")
    
    return df

# ================================
# EXEMPLO DE USO
# ================================

# Definir caminhos
caminho_original = "inter_results/1n_knn.csv"
caminho_aleatorio = "inter_results/1n_random_knn.csv"

# Criar dataset aleatório SEM restrições
dataset_aleatorio = criar_dataset_aleatorio_continuo_sem_restricoes(caminho_original, caminho_aleatorio)

print(f"\n🎉 PROCESSO CONCLUÍDO!")
print(f"📁 Ficheiro original: {caminho_original}")
print(f"📁 Ficheiro aleatório: {caminho_aleatorio}")


📂 Carregando dataset original...
✅ Dataset carregado com 2400 linhas
🎲 Gerando valores TOTALMENTE aleatórios (sem restrições de distância)...
Processando linha 0/2400
Processando linha 1000/2400
Processando linha 2000/2400
💾 Salvando dataset aleatório em: inter_results/1n_random_knn.csv
✅ Dataset aleatório criado com sucesso!
📊 2400 linhas processadas

📈 Estatísticas dos valores aleatórios:
  UAV 1: X=[60.02, 119.99], Y=[0.09, 59.98]
  UAV 2: X=[60.03, 119.97], Y=[0.02, 59.98]
  UAV 3: X=[60.06, 119.96], Y=[0.02, 59.98]
  UAV 4: X=[60.00, 119.98], Y=[0.10, 59.93]

🔍 Distâncias entre UAVs (primeiras 3 linhas - só informativo):
  Linha 0: Distância mínima = 5.17m
  Linha 1: Distância mínima = 25.89m
  Linha 2: Distância mínima = 28.65m

🎉 PROCESSO CONCLUÍDO!
📁 Ficheiro original: inter_results/1n_knn.csv
📁 Ficheiro aleatório: inter_results/1n_random_knn.csv


# ANALISAR TUDO

In [5]:
import numpy as np
import networkx as nx
import pandas as pd

def has_collision_trajectory(initial_positions, final_positions, num_uavs=4, num_timeslots=6, min_distance=20):
    """
    Verifica se há colisões ao longo de toda a trajetória (interpolação linear)
    
    Args:
        initial_positions: Lista de tuplas [(x1,y1), (x2,y2), (x3,y3), (x4,y4)]
        final_positions: Lista de tuplas [(x1,y1), (x2,y2), (x3,y3), (x4,y4)]
        num_uavs: Número de UAVs
        num_timeslots: Número de timeslots para verificar
        min_distance: Distância mínima entre UAVs
    
    Returns:
        bool: True se há colisão em qualquer timeslot, False caso contrário
    """
    
    # Verificar colisões em cada timeslot
    for t in range(num_timeslots):
        # Calcular fator de interpolação
        if num_timeslots == 1:
            alpha = 1.0
        else:
            alpha = t / (num_timeslots - 1)
        
        # Calcular posições interpoladas para este timeslot
        positions_t = []
        for uav in range(num_uavs):
            start_x, start_y = initial_positions[uav]
            end_x, end_y = final_positions[uav]
            
            # Interpolação linear
            x = start_x + alpha * (end_x - start_x)
            y = start_y + alpha * (end_y - start_y)
            
            positions_t.append((x, y))
        
        # Verificar colisões entre todos os pares neste timeslot
        for i in range(num_uavs):
            for j in range(i + 1, num_uavs):
                x1, y1 = positions_t[i]
                x2, y2 = positions_t[j]
                
                distance = np.sqrt((x1 - x2)**2 + (y1 - y2)**2)
                
                if distance < min_distance:
                    return True  # Há colisão neste timeslot
    
    return False  # Sem colisões em nenhum timeslot

def analisar_comunicacoes_interpolado_sem_verificacao_colisoes(initial_positions, final_positions, num_uavs, num_timeslots, jammer_position):
    """
    Analisa comunicações interpolando entre posições inicial e final
    SEM VERIFICAR COLISÕES (para Análise 1)
    """
    
    def criar_best_solution_interpolado(initial_pos, final_pos, num_uavs, num_timeslots):
        """
        Cria best_solution interpolando linearmente entre posições inicial e final
        """
        best_solution = []
        
        for t in range(num_timeslots):
            # Calcular fator de interpolação (0 no início, 1 no final)
            if num_timeslots == 1:
                alpha = 1.0  # Se só há 1 timeslot, usar posição final
            else:
                alpha = t / (num_timeslots - 1)
            
            # Para cada UAV
            for uav in range(num_uavs):
                start_x, start_y = initial_pos[uav]
                end_x, end_y = final_pos[uav]
                
                # Interpolação linear
                x = start_x + alpha * (end_x - start_x)
                y = start_y + alpha * (end_y - start_y)
                
                # Adicionar ao best_solution
                best_solution.extend([x, y])
        
        return best_solution
    
    def avaliar_grafo(comm_matrix):
        G = nx.DiGraph()
        
        # 1. Adicionar arestas bidirecionais com capacidades
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j and comm_matrix[i][j] > 0:
                    G.add_edge(i, j, capacity=comm_matrix[i][j])
        
        # 2. Cálculos de métricas
        metricas = {
            'capacidades': [],
            'caminhos_minimos': {},
            'bottlenecks': {},
            'conectividade': None,
            'links_usados': set(),
            'links_nao_usados': set()
        }
        
        # Preencher métricas
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j:
                    # Calcular caminhos mínimos (1/capacity como peso)
                    try:
                        path = nx.shortest_path(G, source=i, target=j, weight=lambda u, v, d: 1/d['capacity'])
                        capacidade_min = min(G[u][v]['capacity'] for u, v in zip(path[:-1], path[1:]))
                        
                        metricas['caminhos_minimos'][(i,j)] = {
                            'path': path,
                            'capacidade': capacidade_min
                        }
                        
                        # Adicionar links usados
                        for u, v in zip(path[:-1], path[1:]):
                            metricas['links_usados'].add((u, v))
                            metricas['links_usados'].add((v, u))  # Adicionar a aresta reversa
                    except:
                        pass
        
        # Calcular bottlenecks para cada nó
        for node in G.nodes():
            metricas['bottlenecks'][node] = min(
                [d['capacity'] for _, _, d in G.edges(node, data=True)],
                default=0
            )
        
        # Verificar conectividade
        metricas['conectividade'] = nx.is_strongly_connected(G)
        
        # Coletar todas as capacidades
        metricas['capacidades'] = [d['capacity'] for _, _, d in G.edges(data=True)]
        
        # Identificar links não usados
        all_links = {(i, j) for i in range(num_uavs) for j in range(num_uavs) if i != j}
        metricas['links_nao_usados'] = all_links - metricas['links_usados']
        
        return metricas

    # CRIAR BEST_SOLUTION ATRAVÉS DE INTERPOLAÇÃO
    best_solution = criar_best_solution_interpolado(initial_positions, final_positions, num_uavs, num_timeslots)
    
    # SEM VERIFICAÇÃO DE COLISÕES - continuar sempre com a análise
    resultados = []
    todos_c_media = []
    todos_c_min = []
    
    # Analisar cada timeslot
    for t in range(num_timeslots):
        positions = []
        angles = []
        
        for i in range(num_uavs):
            idx_pos = t * num_uavs * 2 + i * 2
            x, y = best_solution[idx_pos:idx_pos+2]
            positions.append(np.array([x, y]))
            
            # Calcular ângulo dinamicamente
            angle = np.degrees(np.arctan2(jammer_position[1] - y, jammer_position[0] - x))
            angles.append(angle)
        
        # Calcular matriz de comunicação
        comm_matrix, _ = calculate_communication_capacity(angles, [1.0]*num_uavs, positions, jammer_position)
        
        # Gerar métricas detalhadas
        metricas = avaliar_grafo(comm_matrix)
        
        # Cálculo do fitness usando apenas links usados
        capacidades_usadas = [comm_matrix[u][v] for u, v in metricas['links_usados'] if u < num_uavs and v < num_uavs]
        if capacidades_usadas:
            C_media_total = np.mean(capacidades_usadas)
            C_min_total = min(capacidades_usadas)
        else:
            C_media_total = 0
            C_min_total = 0
        
        # Adicionar às listas do epoch
        todos_c_media.append(C_media_total)
        todos_c_min.append(C_min_total)
        
        resultados.append({
            'epoch': 1,
            'timeslot': t+1,
            'metricas': metricas,
            'C_media_total': C_media_total,
            'C_min_total': C_min_total
        })
    
    # Calcular fitness final
    alpha, beta = 1.0, 1.0
    fitness_por_timeslot = []
    
    for i in range(len(todos_c_media)):
        C_media = todos_c_media[i]
        C_min = todos_c_min[i]
        fitness_timeslot = (C_media ** alpha) * (C_min ** beta)
        fitness_por_timeslot.append(fitness_timeslot)
    
    # Fitness médio do epoch
    fitness_epoch = np.mean(fitness_por_timeslot) if fitness_por_timeslot else 0
    
    # Calcular também as médias para informação
    media_c_media_epoch = np.mean(todos_c_media) if todos_c_media else 0
    media_c_min_epoch = np.mean(todos_c_min) if todos_c_min else 0
    
    return resultados, fitness_epoch, media_c_media_epoch, media_c_min_epoch

def analisar_comunicacoes_interpolado_final(initial_positions, final_positions, num_uavs, num_timeslots, jammer_position):
    """
    Analisa comunicações interpolando entre posições inicial e final
    VERIFICA COLISÕES APENAS NAS POSIÇÕES FINAIS
    """
    
    def criar_best_solution_interpolado(initial_pos, final_pos, num_uavs, num_timeslots):
        """
        Cria best_solution interpolando linearmente entre posições inicial e final
        """
        best_solution = []
        
        for t in range(num_timeslots):
            # Calcular fator de interpolação (0 no início, 1 no final)
            if num_timeslots == 1:
                alpha = 1.0  # Se só há 1 timeslot, usar posição final
            else:
                alpha = t / (num_timeslots - 1)
            
            # Para cada UAV
            for uav in range(num_uavs):
                start_x, start_y = initial_pos[uav]
                end_x, end_y = final_pos[uav]
                
                # Interpolação linear
                x = start_x + alpha * (end_x - start_x)
                y = start_y + alpha * (end_y - start_y)
                
                # Adicionar ao best_solution
                best_solution.extend([x, y])
        
        return best_solution
    
    def avaliar_grafo(comm_matrix):
        G = nx.DiGraph()
        
        # 1. Adicionar arestas bidirecionais com capacidades
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j and comm_matrix[i][j] > 0:
                    G.add_edge(i, j, capacity=comm_matrix[i][j])
        
        # 2. Cálculos de métricas
        metricas = {
            'capacidades': [],
            'caminhos_minimos': {},
            'bottlenecks': {},
            'conectividade': None,
            'links_usados': set(),
            'links_nao_usados': set()
        }
        
        # Preencher métricas
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j:
                    # Calcular caminhos mínimos (1/capacity como peso)
                    try:
                        path = nx.shortest_path(G, source=i, target=j, weight=lambda u, v, d: 1/d['capacity'])
                        capacidade_min = min(G[u][v]['capacity'] for u, v in zip(path[:-1], path[1:]))
                        
                        metricas['caminhos_minimos'][(i,j)] = {
                            'path': path,
                            'capacidade': capacidade_min
                        }
                        
                        # Adicionar links usados
                        for u, v in zip(path[:-1], path[1:]):
                            metricas['links_usados'].add((u, v))
                            metricas['links_usados'].add((v, u))  # Adicionar a aresta reversa
                    except:
                        pass
        
        # Calcular bottlenecks para cada nó
        for node in G.nodes():
            metricas['bottlenecks'][node] = min(
                [d['capacity'] for _, _, d in G.edges(node, data=True)],
                default=0
            )
        
        # Verificar conectividade
        metricas['conectividade'] = nx.is_strongly_connected(G)
        
        # Coletar todas as capacidades
        metricas['capacidades'] = [d['capacity'] for _, _, d in G.edges(data=True)]
        
        # Identificar links não usados
        all_links = {(i, j) for i in range(num_uavs) for j in range(num_uavs) if i != j}
        metricas['links_nao_usados'] = all_links - metricas['links_usados']
        
        return metricas

    # CRIAR BEST_SOLUTION ATRAVÉS DE INTERPOLAÇÃO
    best_solution = criar_best_solution_interpolado(initial_positions, final_positions, num_uavs, num_timeslots)
    
    # 🆕 VERIFICAR COLISÕES AO LONGO DA TRAJETÓRIA
    tem_colisoes = has_collision_trajectory(initial_positions, final_positions, num_uavs, num_timeslots)
    
    if tem_colisoes:
        return [(0.0, "Solução com colisões nas posições finais")], 0.0, 0.0, 0.0
    
    # Se não há colisões nas posições finais, continuar com a análise normal
    resultados = []
    todos_c_media = []
    todos_c_min = []
    
    # Analisar cada timeslot
    for t in range(num_timeslots):
        positions = []
        angles = []
        
        for i in range(num_uavs):
            idx_pos = t * num_uavs * 2 + i * 2
            x, y = best_solution[idx_pos:idx_pos+2]
            positions.append(np.array([x, y]))
            
            # Calcular ângulo dinamicamente
            angle = np.degrees(np.arctan2(jammer_position[1] - y, jammer_position[0] - x))
            angles.append(angle)
        
        # Calcular matriz de comunicação
        comm_matrix, _ = calculate_communication_capacity(angles, [1.0]*num_uavs, positions, jammer_position)
        
        # Gerar métricas detalhadas
        metricas = avaliar_grafo(comm_matrix)
        
        # Cálculo do fitness usando apenas links usados
        capacidades_usadas = [comm_matrix[u][v] for u, v in metricas['links_usados'] if u < num_uavs and v < num_uavs]
        if capacidades_usadas:
            C_media_total = np.mean(capacidades_usadas)
            C_min_total = min(capacidades_usadas)
        else:
            C_media_total = 0
            C_min_total = 0
        
        # Adicionar às listas do epoch
        todos_c_media.append(C_media_total)
        todos_c_min.append(C_min_total)
        
        resultados.append({
            'epoch': 1,
            'timeslot': t+1,
            'metricas': metricas,
            'C_media_total': C_media_total,
            'C_min_total': C_min_total
        })
    
    # Calcular fitness final
    alpha, beta = 1.0, 1.0
    fitness_por_timeslot = []
    
    for i in range(len(todos_c_media)):
        C_media = todos_c_media[i]
        C_min = todos_c_min[i]
        fitness_timeslot = (C_media ** alpha) * (C_min ** beta)
        fitness_por_timeslot.append(fitness_timeslot)
    
    # Fitness médio do epoch
    fitness_epoch = np.mean(fitness_por_timeslot) if fitness_por_timeslot else 0
    
        # Calcular também as médias para informação
    media_c_media_epoch = np.mean(todos_c_media) if todos_c_media else 0
    media_c_min_epoch = np.mean(todos_c_min) if todos_c_min else 0
    
    return resultados, fitness_epoch, media_c_media_epoch, media_c_min_epoch



In [6]:
def carregar_e_analisar_dataset_final(caminho_dataset, num_uavs=4, num_timeslots=6):
    """
    Carrega o dataset e analisa comunicações para valores reais e preditos
    TRÊS ANÁLISES com verificação de colisões apenas nas posições finais:
    1. Todas as linhas (valores reais)
    2. Apenas sem colisões
    3. Todas as linhas com colisões = 0
    """
    
    # CARREGAR O DATASET
    df = pd.read_csv(caminho_dataset)
    
    # ANÁLISE 1: Todas as linhas (valores reais)
    fitness_reais = []
    fitness_preditos = []
    c_media_reais = []
    c_media_preditos = []
    c_min_reais = []
    c_min_preditos = []
    
    # ANÁLISE 2: Apenas sem colisões
    fitness_reais_sem_colisoes = []
    fitness_preditos_sem_colisoes = []
    c_media_reais_sem_colisoes = []
    c_media_preditos_sem_colisoes = []
    c_min_reais_sem_colisoes = []
    c_min_preditos_sem_colisoes = []
    
    # ANÁLISE 3: Todas as linhas, mas colisões = 0
    fitness_reais_colisoes_zero = []
    fitness_preditos_colisoes_zero = []
    c_media_reais_colisoes_zero = []
    c_media_preditos_colisoes_zero = []
    c_min_reais_colisoes_zero = []
    c_min_preditos_colisoes_zero = []
    
    linhas_sem_colisoes = []
    total_linhas_sem_colisoes = 0
    
    # ANALISAR CADA LINHA DO DATASET
    for index, row in df.iterrows():
        # Extrair posições
        initial_positions = [
            (row['Initial_X1'], row['Initial_Y1']),
            (row['Initial_X2'], row['Initial_Y2']),
            (row['Initial_X3'], row['Initial_Y3']),
            (row['Initial_X4'], row['Initial_Y4'])
        ]
        
        final_positions_preditas = [
            (row['Predicted_Final_X1'], row['Predicted_Final_Y1']),
            (row['Predicted_Final_X2'], row['Predicted_Final_Y2']),
            (row['Predicted_Final_X3'], row['Predicted_Final_Y3']),
            (row['Predicted_Final_X4'], row['Predicted_Final_Y4'])
        ]
        
        final_positions_reais = [
            (row['Real_Final_X1'], row['Real_Final_Y1']),
            (row['Real_Final_X2'], row['Real_Final_Y2']),
            (row['Real_Final_X3'], row['Real_Final_Y3']),
            (row['Real_Final_X4'], row['Real_Final_Y4'])
        ]
        
        jammer_position = [row['Jammer_X'], row['Jammer_Y']]
        
        # VERIFICAR COLISÕES APENAS NAS POSIÇÕES FINAIS
        # VERIFICAR COLISÕES AO LONGO DA TRAJETÓRIA
        tem_colisoes_predito = has_collision_trajectory(initial_positions, final_positions_preditas, num_uavs, num_timeslots)
        tem_colisoes_real = has_collision_trajectory(initial_positions, final_positions_reais, num_uavs, num_timeslots)

        
        # CALCULAR VALORES REAIS DE COMUNICAÇÃO (mesmo com colisões) - USANDO FUNÇÃO SEM VERIFICAÇÃO
        resultados_pred, fitness_pred, c_media_pred, c_min_pred = analisar_comunicacoes_interpolado_sem_verificacao_colisoes(
            initial_positions=initial_positions,
            final_positions=final_positions_preditas,
            num_uavs=num_uavs,
            num_timeslots=num_timeslots,
            jammer_position=jammer_position
        )
        
        resultados_real, fitness_real, c_media_real, c_min_real = analisar_comunicacoes_interpolado_sem_verificacao_colisoes(
            initial_positions=initial_positions,
            final_positions=final_positions_reais,
            num_uavs=num_uavs,
            num_timeslots=num_timeslots,
            jammer_position=jammer_position
        )
        
        # ANÁLISE 1: Armazenar valores reais (independente de colisões)
        fitness_preditos.append(fitness_pred)
        fitness_reais.append(fitness_real)
        c_media_preditos.append(c_media_pred)
        c_media_reais.append(c_media_real)
        c_min_preditos.append(c_min_pred)
        c_min_reais.append(c_min_real)
        
        # ANÁLISE 2: Se não há colisões, adicionar às listas especiais
        if not tem_colisoes_predito:
            fitness_preditos_sem_colisoes.append(fitness_pred)
            fitness_reais_sem_colisoes.append(fitness_real)
            c_media_preditos_sem_colisoes.append(c_media_pred)
            c_media_reais_sem_colisoes.append(c_media_real)
            c_min_preditos_sem_colisoes.append(c_min_pred)
            c_min_reais_sem_colisoes.append(c_min_real)
            
            linhas_sem_colisoes.append(index + 1)
            total_linhas_sem_colisoes += 1
        
        # ANÁLISE 3: Colisões = 0
        # Para preditos
        if tem_colisoes_predito:
            fitness_preditos_colisoes_zero.append(0.0)
            c_media_preditos_colisoes_zero.append(0.0)
            c_min_preditos_colisoes_zero.append(0.0)
        else:
            fitness_preditos_colisoes_zero.append(fitness_pred)
            c_media_preditos_colisoes_zero.append(c_media_pred)
            c_min_preditos_colisoes_zero.append(c_min_pred)
        
        # Para reais
        if tem_colisoes_real:
            fitness_reais_colisoes_zero.append(0.0)
            c_media_reais_colisoes_zero.append(0.0)
            c_min_reais_colisoes_zero.append(0.0)
        else:
            fitness_reais_colisoes_zero.append(fitness_real)
            c_media_reais_colisoes_zero.append(c_media_real)
            c_min_reais_colisoes_zero.append(c_min_real)
    
    # CALCULAR MÉDIAS
    media_fitness_real_1 = np.mean(fitness_reais)
    media_fitness_predito_1 = np.mean(fitness_preditos)
    media_c_media_real_1 = np.mean(c_media_reais)
    media_c_media_predito_1 = np.mean(c_media_preditos)
    media_c_min_real_1 = np.mean(c_min_reais)
    media_c_min_predito_1 = np.mean(c_min_preditos)
    
    media_fitness_real_3 = np.mean(fitness_reais_colisoes_zero)
    media_fitness_predito_3 = np.mean(fitness_preditos_colisoes_zero)
    media_c_media_real_3 = np.mean(c_media_reais_colisoes_zero)
    media_c_media_predito_3 = np.mean(c_media_preditos_colisoes_zero)
    media_c_min_real_3 = np.mean(c_min_reais_colisoes_zero)
    media_c_min_predito_3 = np.mean(c_min_preditos_colisoes_zero)
    
    # ANÁLISE 1: RESULTADOS PARA TODAS AS LINHAS
    print(f"============================================================")
    print(f"ANÁLISE 1 - TODAS AS LINHAS (valores reais de comunicação)")
    print(f"============================================================")
    print(f"🎯 FITNESS: Real={media_fitness_real_1:.4f} | Predito={media_fitness_predito_1:.4f} | Diff={abs(media_fitness_real_1 - media_fitness_predito_1):.4f}")
    print(f"📊 C_MÉDIA: Real={media_c_media_real_1:.2f} | Predito={media_c_media_predito_1:.2f} | Diff={abs(media_c_media_real_1 - media_c_media_predito_1):.2f}")
    print(f"📉 C_MIN: Real={media_c_min_real_1:.2f} | Predito={media_c_min_predito_1:.2f} | Diff={abs(media_c_min_real_1 - media_c_min_predito_1):.2f}")
    
    # ANÁLISE 2: APENAS SEM COLISÕES
    if total_linhas_sem_colisoes > 0:
        media_fitness_real_2 = np.mean(fitness_reais_sem_colisoes)
        media_fitness_predito_2 = np.mean(fitness_preditos_sem_colisoes)
        media_c_media_real_2 = np.mean(c_media_reais_sem_colisoes)
        media_c_media_predito_2 = np.mean(c_media_preditos_sem_colisoes)
        media_c_min_real_2 = np.mean(c_min_reais_sem_colisoes)
        media_c_min_predito_2 = np.mean(c_min_preditos_sem_colisoes)
        
        print(f"\n======================================================================")
        print(f"ANÁLISE 2 - APENAS LINHAS SEM COLISÕES ({total_linhas_sem_colisoes}/{len(df)} linhas)")
        print(f"======================================================================")
        print(f"🎯 FITNESS: Real={media_fitness_real_2:.4f} | Predito={media_fitness_predito_2:.4f} | Diff={abs(media_fitness_real_2 - media_fitness_predito_2):.4f}")
        print(f"📊 C_MÉDIA: Real={media_c_media_real_2:.2f} | Predito={media_c_media_predito_2:.2f} | Diff={abs(media_c_media_real_2 - media_c_media_predito_2):.2f}")
        print(f"📉 C_MIN: Real={media_c_min_real_2:.2f} | Predito={media_c_min_predito_2:.2f} | Diff={abs(media_c_min_real_2 - media_c_min_predito_2):.2f}")
    
    # ANÁLISE 3: COLISÕES = 0
    print(f"\n======================================================================")
    print(f"ANÁLISE 3 - TODAS AS LINHAS (colisões = 0)")
    print(f"======================================================================")
    print(f"🎯 FITNESS: Real={media_fitness_real_3:.4f} | Predito={media_fitness_predito_3:.4f} | Diff={abs(media_fitness_real_3 - media_fitness_predito_3):.4f}")
    print(f"📊 C_MÉDIA: Real={media_c_media_real_3:.2f} | Predito={media_c_media_predito_3:.2f} | Diff={abs(media_c_media_real_3 - media_c_media_predito_3):.2f}")
    print(f"📉 C_MIN: Real={media_c_min_real_3:.2f} | Predito={media_c_min_predito_3:.2f} | Diff={abs(media_c_min_real_3 - media_c_min_predito_3):.2f}")

# quero fazer a anlise de todos os datasets gerados (1n_knn.csv, 2n_knn.csv, ..., 10n_knn.csv)
# mas tem de estar identificado o n no nome do ficheiro

if __name__ == "__main__":
    
    print(f"\n\n\n🔹🔹🔹 RANDOM 🔹🔹🔹\n")
    caminho_dataset = f"inter_results/1n_random_knn.csv"
    carregar_e_analisar_dataset_final(caminho_dataset=caminho_dataset, num_uavs=4, num_timeslots=6)




🔹🔹🔹 RANDOM 🔹🔹🔹

ANÁLISE 1 - TODAS AS LINHAS (valores reais de comunicação)
🎯 FITNESS: Real=18637252.4897 | Predito=15112202.0392 | Diff=3525050.4505
📊 C_MÉDIA: Real=6214.50 | Predito=6637.71 | Diff=423.21
📉 C_MIN: Real=2739.45 | Predito=1829.19 | Diff=910.26

ANÁLISE 2 - APENAS LINHAS SEM COLISÕES (26/2400 linhas)
🎯 FITNESS: Real=17241872.9608 | Predito=8594259.4870 | Diff=8647613.4738
📊 C_MÉDIA: Real=5859.04 | Predito=5312.59 | Diff=546.45
📉 C_MIN: Real=2722.66 | Predito=1484.38 | Diff=1238.27

ANÁLISE 3 - TODAS AS LINHAS (colisões = 0)
🎯 FITNESS: Real=17647643.4736 | Predito=93104.4778 | Diff=17554538.9958
📊 C_MÉDIA: Real=5895.72 | Predito=57.55 | Diff=5838.17
📉 C_MIN: Real=2636.21 | Predito=16.08 | Diff=2620.13


In [7]:
def carregar_e_analisar_dataset_final(caminho_dataset, num_uavs=4, num_timeslots=6):
    """
    Carrega o dataset e analisa comunicações para valores reais e preditos
    TRÊS ANÁLISES com verificação de colisões apenas nas posições finais:
    1. Todas as linhas (valores reais)
    2. Apenas sem colisões
    3. Todas as linhas com colisões = 0
    """
    
    # CARREGAR O DATASET
    df = pd.read_csv(caminho_dataset)
    
    # ANÁLISE 1: Todas as linhas (valores reais)
    fitness_reais = []
    fitness_preditos = []
    c_media_reais = []
    c_media_preditos = []
    c_min_reais = []
    c_min_preditos = []
    
    # ANÁLISE 2: Apenas sem colisões
    fitness_reais_sem_colisoes = []
    fitness_preditos_sem_colisoes = []
    c_media_reais_sem_colisoes = []
    c_media_preditos_sem_colisoes = []
    c_min_reais_sem_colisoes = []
    c_min_preditos_sem_colisoes = []
    
    # ANÁLISE 3: Todas as linhas, mas colisões = 0
    fitness_reais_colisoes_zero = []
    fitness_preditos_colisoes_zero = []
    c_media_reais_colisoes_zero = []
    c_media_preditos_colisoes_zero = []
    c_min_reais_colisoes_zero = []
    c_min_preditos_colisoes_zero = []
    
    linhas_sem_colisoes = []
    total_linhas_sem_colisoes = 0
    
    # ANALISAR CADA LINHA DO DATASET
    for index, row in df.iterrows():
        # Extrair posições
        initial_positions = [
            (row['Initial_X1'], row['Initial_Y1']),
            (row['Initial_X2'], row['Initial_Y2']),
            (row['Initial_X3'], row['Initial_Y3']),
            (row['Initial_X4'], row['Initial_Y4'])
        ]
        
        final_positions_preditas = [
            (row['Predicted_Final_X1'], row['Predicted_Final_Y1']),
            (row['Predicted_Final_X2'], row['Predicted_Final_Y2']),
            (row['Predicted_Final_X3'], row['Predicted_Final_Y3']),
            (row['Predicted_Final_X4'], row['Predicted_Final_Y4'])
        ]
        
        final_positions_reais = [
            (row['Real_Final_X1'], row['Real_Final_Y1']),
            (row['Real_Final_X2'], row['Real_Final_Y2']),
            (row['Real_Final_X3'], row['Real_Final_Y3']),
            (row['Real_Final_X4'], row['Real_Final_Y4'])
        ]
        
        jammer_position = [row['Jammer_X'], row['Jammer_Y']]
        
        # VERIFICAR COLISÕES APENAS NAS POSIÇÕES FINAIS
        # VERIFICAR COLISÕES AO LONGO DA TRAJETÓRIA
        tem_colisoes_predito = has_collision_trajectory(initial_positions, final_positions_preditas, num_uavs, num_timeslots)
        tem_colisoes_real = has_collision_trajectory(initial_positions, final_positions_reais, num_uavs, num_timeslots)

        
        # CALCULAR VALORES REAIS DE COMUNICAÇÃO (mesmo com colisões) - USANDO FUNÇÃO SEM VERIFICAÇÃO
        resultados_pred, fitness_pred, c_media_pred, c_min_pred = analisar_comunicacoes_interpolado_sem_verificacao_colisoes(
            initial_positions=initial_positions,
            final_positions=final_positions_preditas,
            num_uavs=num_uavs,
            num_timeslots=num_timeslots,
            jammer_position=jammer_position
        )
        
        resultados_real, fitness_real, c_media_real, c_min_real = analisar_comunicacoes_interpolado_sem_verificacao_colisoes(
            initial_positions=initial_positions,
            final_positions=final_positions_reais,
            num_uavs=num_uavs,
            num_timeslots=num_timeslots,
            jammer_position=jammer_position
        )
        
        # ANÁLISE 1: Armazenar valores reais (independente de colisões)
        fitness_preditos.append(fitness_pred)
        fitness_reais.append(fitness_real)
        c_media_preditos.append(c_media_pred)
        c_media_reais.append(c_media_real)
        c_min_preditos.append(c_min_pred)
        c_min_reais.append(c_min_real)
        
        # ANÁLISE 2: Se não há colisões, adicionar às listas especiais
        if not tem_colisoes_predito:
            fitness_preditos_sem_colisoes.append(fitness_pred)
            fitness_reais_sem_colisoes.append(fitness_real)
            c_media_preditos_sem_colisoes.append(c_media_pred)
            c_media_reais_sem_colisoes.append(c_media_real)
            c_min_preditos_sem_colisoes.append(c_min_pred)
            c_min_reais_sem_colisoes.append(c_min_real)
            
            linhas_sem_colisoes.append(index + 1)
            total_linhas_sem_colisoes += 1
        
        # ANÁLISE 3: Colisões = 0
        # Para preditos
        if tem_colisoes_predito:
            fitness_preditos_colisoes_zero.append(0.0)
            c_media_preditos_colisoes_zero.append(0.0)
            c_min_preditos_colisoes_zero.append(0.0)
        else:
            fitness_preditos_colisoes_zero.append(fitness_pred)
            c_media_preditos_colisoes_zero.append(c_media_pred)
            c_min_preditos_colisoes_zero.append(c_min_pred)
        
        # Para reais
        if tem_colisoes_real:
            fitness_reais_colisoes_zero.append(0.0)
            c_media_reais_colisoes_zero.append(0.0)
            c_min_reais_colisoes_zero.append(0.0)
        else:
            fitness_reais_colisoes_zero.append(fitness_real)
            c_media_reais_colisoes_zero.append(c_media_real)
            c_min_reais_colisoes_zero.append(c_min_real)
    
    # CALCULAR MÉDIAS
    media_fitness_real_1 = np.mean(fitness_reais)
    media_fitness_predito_1 = np.mean(fitness_preditos)
    media_c_media_real_1 = np.mean(c_media_reais)
    media_c_media_predito_1 = np.mean(c_media_preditos)
    media_c_min_real_1 = np.mean(c_min_reais)
    media_c_min_predito_1 = np.mean(c_min_preditos)
    
    media_fitness_real_3 = np.mean(fitness_reais_colisoes_zero)
    media_fitness_predito_3 = np.mean(fitness_preditos_colisoes_zero)
    media_c_media_real_3 = np.mean(c_media_reais_colisoes_zero)
    media_c_media_predito_3 = np.mean(c_media_preditos_colisoes_zero)
    media_c_min_real_3 = np.mean(c_min_reais_colisoes_zero)
    media_c_min_predito_3 = np.mean(c_min_preditos_colisoes_zero)
    
    # ANÁLISE 1: RESULTADOS PARA TODAS AS LINHAS
    print(f"============================================================")
    print(f"ANÁLISE 1 - TODAS AS LINHAS (valores reais de comunicação)")
    print(f"============================================================")
    print(f"🎯 FITNESS: Real={media_fitness_real_1:.4f} | Predito={media_fitness_predito_1:.4f} | Diff={abs(media_fitness_real_1 - media_fitness_predito_1):.4f}")
    print(f"📊 C_MÉDIA: Real={media_c_media_real_1:.2f} | Predito={media_c_media_predito_1:.2f} | Diff={abs(media_c_media_real_1 - media_c_media_predito_1):.2f}")
    print(f"📉 C_MIN: Real={media_c_min_real_1:.2f} | Predito={media_c_min_predito_1:.2f} | Diff={abs(media_c_min_real_1 - media_c_min_predito_1):.2f}")
    
    # ANÁLISE 2: APENAS SEM COLISÕES
    if total_linhas_sem_colisoes > 0:
        media_fitness_real_2 = np.mean(fitness_reais_sem_colisoes)
        media_fitness_predito_2 = np.mean(fitness_preditos_sem_colisoes)
        media_c_media_real_2 = np.mean(c_media_reais_sem_colisoes)
        media_c_media_predito_2 = np.mean(c_media_preditos_sem_colisoes)
        media_c_min_real_2 = np.mean(c_min_reais_sem_colisoes)
        media_c_min_predito_2 = np.mean(c_min_preditos_sem_colisoes)
        
        print(f"\n======================================================================")
        print(f"ANÁLISE 2 - APENAS LINHAS SEM COLISÕES ({total_linhas_sem_colisoes}/{len(df)} linhas)")
        print(f"======================================================================")
        print(f"🎯 FITNESS: Real={media_fitness_real_2:.4f} | Predito={media_fitness_predito_2:.4f} | Diff={abs(media_fitness_real_2 - media_fitness_predito_2):.4f}")
        print(f"📊 C_MÉDIA: Real={media_c_media_real_2:.2f} | Predito={media_c_media_predito_2:.2f} | Diff={abs(media_c_media_real_2 - media_c_media_predito_2):.2f}")
        print(f"📉 C_MIN: Real={media_c_min_real_2:.2f} | Predito={media_c_min_predito_2:.2f} | Diff={abs(media_c_min_real_2 - media_c_min_predito_2):.2f}")
    
    # ANÁLISE 3: COLISÕES = 0
    print(f"\n======================================================================")
    print(f"ANÁLISE 3 - TODAS AS LINHAS (colisões = 0)")
    print(f"======================================================================")
    print(f"🎯 FITNESS: Real={media_fitness_real_3:.4f} | Predito={media_fitness_predito_3:.4f} | Diff={abs(media_fitness_real_3 - media_fitness_predito_3):.4f}")
    print(f"📊 C_MÉDIA: Real={media_c_media_real_3:.2f} | Predito={media_c_media_predito_3:.2f} | Diff={abs(media_c_media_real_3 - media_c_media_predito_3):.2f}")
    print(f"📉 C_MIN: Real={media_c_min_real_3:.2f} | Predito={media_c_min_predito_3:.2f} | Diff={abs(media_c_min_real_3 - media_c_min_predito_3):.2f}")

# quero fazer a anlise de todos os datasets gerados (1n_knn.csv, 2n_knn.csv, ..., 10n_knn.csv)
# mas tem de estar identificado o n no nome do ficheiro

if __name__ == "__main__":
    for n in range(1, 16):
        print(f"\n\n\n🔹🔹🔹 ANÁLISE DO DATASET COM {n} VIZINHOS 🔹🔹🔹\n")
        caminho_dataset = f"inter_results/{n}n_knn.csv"
        carregar_e_analisar_dataset_final(caminho_dataset=caminho_dataset, num_uavs=4, num_timeslots=6)








🔹🔹🔹 ANÁLISE DO DATASET COM 1 VIZINHOS 🔹🔹🔹

ANÁLISE 1 - TODAS AS LINHAS (valores reais de comunicação)
🎯 FITNESS: Real=18637252.4897 | Predito=27112785.6400 | Diff=8475533.1503
📊 C_MÉDIA: Real=6214.50 | Predito=7341.37 | Diff=1126.87
📉 C_MIN: Real=2739.45 | Predito=3398.78 | Diff=659.32

ANÁLISE 2 - APENAS LINHAS SEM COLISÕES (1039/2400 linhas)
🎯 FITNESS: Real=17963774.8678 | Predito=24442232.2010 | Diff=6478457.3331
📊 C_MÉDIA: Real=6239.57 | Predito=7410.39 | Diff=1170.81
📉 C_MIN: Real=2611.46 | Predito=3059.04 | Diff=447.58

ANÁLISE 3 - TODAS AS LINHAS (colisões = 0)
🎯 FITNESS: Real=17647643.4736 | Predito=10581449.6903 | Diff=7066193.7833
📊 C_MÉDIA: Real=5895.72 | Predito=3208.08 | Diff=2687.64
📉 C_MIN: Real=2636.21 | Predito=1324.31 | Diff=1311.90



🔹🔹🔹 ANÁLISE DO DATASET COM 2 VIZINHOS 🔹🔹🔹

ANÁLISE 1 - TODAS AS LINHAS (valores reais de comunicação)
🎯 FITNESS: Real=18637252.4897 | Predito=29821012.0880 | Diff=11183759.5983
📊 C_MÉDIA: Real=6214.50 | Predito=7416.43 | Diff=1201.93

# MED

In [8]:
import pandas as pd
import numpy as np
import os
import glob

def calculate_mean_euclidean_distance_single(dataset_file):
    """
    Calcula a Mean Euclidean Distance para um único ficheiro
    
    Args:
        dataset_file: Caminho do ficheiro CSV
    
    Returns:
        dict: Estatísticas da distância euclidiana
    """
    
    try:
        # Carregar dataset
        df = pd.read_csv(dataset_file)
        
        # Lista para armazenar todas as distâncias
        all_distances = []
        
        # Calcular distância euclidiana para cada linha e cada UAV
        for index, row in df.iterrows():
            for uav in range(1, 5):  # UAVs 1, 2, 3, 4
                # Posições preditas
                pred_x = row[f'Predicted_Final_X{uav}']
                pred_y = row[f'Predicted_Final_Y{uav}']
                
                # Posições reais
                real_x = row[f'Real_Final_X{uav}']
                real_y = row[f'Real_Final_Y{uav}']
                
                # Calcular distância euclidiana
                distance = np.sqrt((pred_x - real_x)**2 + (pred_y - real_y)**2)
                all_distances.append(distance)
        
        # Calcular estatísticas
        results = {
            'filename': os.path.basename(dataset_file),
            'total_predictions': len(all_distances),
            'total_lines': len(df),
            'mean_distance': np.mean(all_distances),
            'std_distance': np.std(all_distances),
            'min_distance': np.min(all_distances),
            'max_distance': np.max(all_distances),
            'median_distance': np.median(all_distances)
        }
        
        return results
        
    except Exception as e:
        print(f"❌ Erro ao processar {dataset_file}: {e}")
        return None

def analyze_multiple_files(file_list=None, directory=None, pattern="*.csv"):
    """
    Analisa múltiplos ficheiros e calcula Mean Euclidean Distance
    
    Args:
        file_list: Lista de caminhos de ficheiros (opcional)
        directory: Diretório para procurar ficheiros (opcional)
        pattern: Padrão de ficheiros a procurar (default: "*.csv")
    """
    
    # Determinar lista de ficheiros
    if file_list:
        files = file_list
    elif directory:
        files = glob.glob(os.path.join(directory, pattern))
    else:
        print("❌ Deve fornecer file_list ou directory")
        return
    
    if not files:
        print("❌ Nenhum ficheiro encontrado")
        return
    
    print(f"🔍 Encontrados {len(files)} ficheiros para analisar")
    print("="*80)
    
    # Analisar cada ficheiro
    results = []
    
    for file_path in files:
        print(f"📊 Analisando: {os.path.basename(file_path)}")
        result = calculate_mean_euclidean_distance_single(file_path)
        
        if result:
            results.append(result)
            print(f"   ✅ Mean Euclidean Distance: {result['mean_distance']:.4f}")
        else:
            print(f"   ❌ Falhou")
        print()
    
    # Mostrar resumo comparativo
    if results:
        print("="*80)
        print("📈 RESUMO COMPARATIVO - MEAN EUCLIDEAN DISTANCE")
        print("="*80)
        
        # Cabeçalho
        print(f"{'Ficheiro':<40} {'Mean Dist':<12} {'Std':<10} {'Min':<10} {'Max':<10} {'Linhas':<8}")
        print("-" * 90)
        
        # Resultados
        for result in results:
            filename = result['filename'][:37] + "..." if len(result['filename']) > 40 else result['filename']
            print(f"{filename:<40} {result['mean_distance']:<12.4f} {result['std_distance']:<10.4f} "
                  f"{result['min_distance']:<10.4f} {result['max_distance']:<10.4f} {result['total_lines']:<8}")
        
        # Encontrar melhor e pior
        best = min(results, key=lambda x: x['mean_distance'])
        worst = max(results, key=lambda x: x['mean_distance'])
        
        print("\n" + "="*50)
        print("🏆 RANKING:")
        print(f"   🥇 MELHOR: {best['filename']} (Mean: {best['mean_distance']:.4f})")
        print(f"   🥉 PIOR:   {worst['filename']} (Mean: {worst['mean_distance']:.4f})")
        print(f"   📊 DIFERENÇA: {worst['mean_distance'] - best['mean_distance']:.4f}")
        
        return results
    
    else:
        print("❌ Nenhum ficheiro foi processado com sucesso")
        return None

# Função de conveniência para usar facilmente
def quick_analysis(files):
    """
    Análise rápida de uma lista de ficheiros
    
    Args:
        files: Lista de caminhos de ficheiros
    """
    return analyze_multiple_files(file_list=files)

# Executar
if __name__ == "__main__":
    
    # OPÇÃO 1: Lista específica de ficheiros
    files_to_analyze = [
        "inter_results/1n_knn.csv",
        "inter_results/2n_knn.csv",
        "inter_results/3n_knn.csv",
        "inter_results/4n_knn.csv",
        "inter_results/5n_knn.csv",
        "inter_results/6n_knn.csv",
        "inter_results/7n_knn.csv",
        "inter_results/8n_knn.csv",
        "inter_results/9n_knn.csv",
        "inter_results/10n_knn.csv",
        "inter_results/11n_knn.csv",
        "inter_results/12n_knn.csv",
        "inter_results/13n_knn.csv",
        "inter_results/14n_knn.csv",
        "inter_results/15n_knn.csv",
        "inter_results/1n_random_knn.csv",
        
    ]
    
    print("🚀 ANÁLISE DE MEAN EUCLIDEAN DISTANCE")
    print("="*80)
    
    results = quick_analysis(files_to_analyze)
    
    # OPÇÃO 2: Todos os ficheiros de um diretório
    # results = analyze_multiple_files(directory="class_results", pattern="*.csv")
    
    # OPÇÃO 3: Ficheiro individual
    # result = calculate_mean_euclidean_distance_single("inter_results/1n_knn.csv")
    # print(f"Mean Distance: {result['mean_distance']:.4f}")


🚀 ANÁLISE DE MEAN EUCLIDEAN DISTANCE
🔍 Encontrados 16 ficheiros para analisar
📊 Analisando: 1n_knn.csv
   ✅ Mean Euclidean Distance: 25.4708

📊 Analisando: 2n_knn.csv
   ✅ Mean Euclidean Distance: 21.9151

📊 Analisando: 3n_knn.csv
   ✅ Mean Euclidean Distance: 21.0912

📊 Analisando: 4n_knn.csv
   ✅ Mean Euclidean Distance: 20.2888

📊 Analisando: 5n_knn.csv
   ✅ Mean Euclidean Distance: 19.8281

📊 Analisando: 6n_knn.csv
   ✅ Mean Euclidean Distance: 19.6121

📊 Analisando: 7n_knn.csv
   ✅ Mean Euclidean Distance: 19.3617

📊 Analisando: 8n_knn.csv
   ✅ Mean Euclidean Distance: 19.1408

📊 Analisando: 9n_knn.csv
   ✅ Mean Euclidean Distance: 19.0779

📊 Analisando: 10n_knn.csv
   ✅ Mean Euclidean Distance: 18.9209

📊 Analisando: 11n_knn.csv
   ✅ Mean Euclidean Distance: 18.8308

📊 Analisando: 12n_knn.csv
   ✅ Mean Euclidean Distance: 18.8373

📊 Analisando: 13n_knn.csv
   ✅ Mean Euclidean Distance: 18.8003

📊 Analisando: 14n_knn.csv
   ✅ Mean Euclidean Distance: 18.6821

📊 Analisando: 15n_knn